# TRAPPIST-1 K2 Campaign 12 — Exoplanet Transit Analysis 

### Setup

Dependencies are listed in `requirements.txt`.

Install locally with:

```bash
pip install -r requirements.txt
```

---

### Data

Required input files:

- K2 light curve FITS file
- K2 Target Pixel File (TPF)

See `data/README.md` for download instructions.

---

### Path Configuration

Default workflow uses **Google Colab + Google Drive**.

Set the base directory in the configuration cell:

### Colab / Drive (default)

```python
DRIVE = "/content/drive/MyDrive/Colab Notebooks/Your_Folder_Name"
```

### Local execution

Comment out Drive setup and use:

```python
DRIVE = "Path/to/local/folder"
```

---

### Runtime Notes

- **TLS searches** may require significant runtime.
- **CETRA sections** requires GPU execution.

For Google Colab GPU:

```text
Runtime → Change runtime type → T4 GPU
```

### Recommended Workflow for Repeated Runs

Some sections are computationally expensive (particularly TLS and CETRA).
For faster repeated execution:

1. Run the notebook once with all cells enabled.
2. Save generated data/products to `BASE_DIR`.
3. For subsequent runs, comment out expensive recomputation cells and load the saved outputs instead.

This can substantially reduce runtime for long-running sections.

# Phase 1 - Exploratory Data Analysis

In [ ]:
# #Uncomment and Run only when using Google Colab
# !pip install Astroquery
# !pip install celerite2
# !pip install transitleastsquares
# !pip install pycuda
# !pip install cetra
# !pip install lightkurve

In [ ]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
import celerite2
from celerite2 import terms
from transitleastsquares import transitleastsquares, cleaned_array
import cetra
from cetra import LightCurve, TransitDetector
import pycuda.autoinit
import pycuda.driver as cuda
import lightkurve as lk

In [ ]:
try:
    # For Google Colab + Google Drive Workflow
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = "/content/drive/MyDrive/Colab Notebooks/Your_Folder_Name"   # Add the name of your folder
except ImportError:
    print("Running outside Google Colab.")

# Optional local execution:
# Comment the above lines of Google Environment and uncomment below for Local Envionment
# DRIVE = 'Path/to/local/folder'

In [ ]:
k2 = fits.open(f'{DRIVE}/k2.fits')
k2.info()

In [ ]:
k2[1].columns.names

In [ ]:
lc_k2 = k2[1].data
df1 = pd.DataFrame(np.array(lc_k2))
print(df1.shape)
df1.head()

In [ ]:
df1.describe()

In [ ]:
plt.figure(figsize=(14,4))
plt.subplot(1,3,1)
plt.suptitle('K2 Rolling Effect')
plt.subplot(1,3,1)
plt.scatter(df1['TIME'], df1['SAP_FLUX'], color ='red', label = 'SAP FLUX', s = 0.5, alpha = 0.3)
plt.xlabel('Time')
plt.ylabel('SAP_FLUX')
plt.title('SAP_FLUX')
plt.subplot(1,3,2)
plt.scatter(df1['TIME'], df1['POS_CORR1'], color ='blue', label = 'POS CORR1', s = 0.5, alpha = 0.3)
plt.xlabel('Time')
plt.ylabel('POS_CORR1')
plt.title('POS_CORR1')
plt.subplot(1,3,3)
plt.scatter(df1['TIME'], df1['POS_CORR2'], color ='green', label = 'POS CORR2', s = 0.5, alpha = 0.3)
plt.xlabel('Time')
plt.ylabel('POS_CORR2')
plt.title('POS_CORR2')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14,4))
plt.subplot(1,3,1)
plt.suptitle('K2 Rolling Effect 2910-2912 BKJD ')
plt.subplot(1,3,1)
plt.scatter(df1['TIME'], df1['SAP_FLUX'], color ='red', label = 'SAP FLUX', s = 0.5, alpha = 0.3)
plt.xlim(2910,2912)
plt.xlabel('Time')
plt.ylabel('SAP_FLUX')
plt.title('SAP_FLUX')
plt.subplot(1,3,2)
plt.scatter(df1['TIME'], df1['POS_CORR1'], color ='blue', label = 'POS CORR1', s = 0.5, alpha = 0.3)
plt.xlim(2910,2912)
plt.xlabel('Time')
plt.ylabel('POS_CORR1')
plt.title('POS_CORR1')
plt.subplot(1,3,3)
plt.scatter(df1['TIME'], df1['POS_CORR2'], color ='green', label = 'POS CORR2', s = 0.5, alpha = 0.3)
plt.xlim(2910,2912)
plt.xlabel('Time')
plt.ylabel('POS_CORR2')
plt.title('POS_CORR2')
plt.tight_layout()
plt.show()

In [ ]:
print((df1['SAP_QUALITY'] == 0).sum())
flag = df1['SAP_QUALITY'] != 0
df_flag = df1[flag]
print(df_flag['SAP_QUALITY'].value_counts().head(10))

In [ ]:
print(df1['SAP_FLUX'].isna().sum())
print(df1['POS_CORR1'].isna().sum())
len(df1[(df1['SAP_QUALITY'] == 0) & (df1['SAP_FLUX'].isna())])

# Phase 2 - Data Cleaning

In [ ]:
# Sector Processing Function
def process_sector(filepath, verbose = True):
  try:
      # Opening fits and Extracting Columns
      k2 = fits.open(filepath)
      data = k2[1].data
      k2.close()

      # Build DataFrame
      df = pd.DataFrame(np.array(data))[['TIME', 'SAP_FLUX', 'SAP_FLUX_ERR', 'SAP_QUALITY', 'POS_CORR1', 'POS_CORR2']]

      # Quality Filter and dropna
      df_clean = df[df['SAP_QUALITY'] == 0].copy()
      df_clean = df_clean.reset_index(drop=True)

      if verbose:
        print(f"Raw points: {len(df)}")
        print(f"Clean points after quality filter: {len(df_clean)}")

      # Normalizing DataFrames
      median1 = np.median(df_clean['SAP_FLUX'])
      NORM_FLUX = df_clean['SAP_FLUX'] / median1
      df_clean['NORM_FLUX'] = NORM_FLUX

      NORM_FLUX_ERR = df_clean['SAP_FLUX_ERR']/median1
      df_clean['NORM_FLUX_ERR'] = NORM_FLUX_ERR

      if verbose:
        print(f"Flux median: {median1:.2f}")
        print(f"Norm flux range: {df_clean['NORM_FLUX'].min():.3f} to {df_clean['NORM_FLUX'].max():.3f}")

      # Sigma Clipping
      flux_med = np.median(df_clean['NORM_FLUX'])
      flux_std = np.std(df_clean['NORM_FLUX'])

      mask = (df_clean['NORM_FLUX'] > flux_med - 3*flux_std) & (df_clean['NORM_FLUX'] < flux_med + 3*flux_std) # Boolean Mask
      df_filtered = df_clean[mask]
      filtered_std = np.std(df_filtered['NORM_FLUX'])

      if verbose:
        print(f"Points after sigma clip: {len(df_filtered)}")
        print(f"Outliers removed: {len(df_clean) - len(df_filtered)}")
        print(f"Filtered std: {filtered_std:.5f}")

      # Gap detection + split into chunks
      time_diff = df_filtered['TIME'].diff()
      gap_indices = np.where(time_diff > 0.1)[0]
      boundaries = [0] + list(gap_indices) + [len(df_filtered)]
      chunks = []
      for i in range(len(boundaries) - 1):
          chunk = df_filtered.iloc[boundaries[i]:boundaries[i+1]].copy()
          chunks.append(chunk)

      if verbose:
        print(f"Number of chunks: {len(chunks)}")
        print(f"Chunk sizes: {[len(c) for c in chunks]}")

      # Savitzky-Golay per chunk (window=401, order=2)
      for chunk in chunks:
        window = min(401, len(chunk))
        if window % 2 == 0:  # must be odd
            window -= 1
        if window >= 3:  # minimum valid window
            trend = savgol_filter(chunk['NORM_FLUX'], window, 2)
            chunk['DETRENDED_FLUX'] = chunk['NORM_FLUX'] / trend
        else:
            chunk['DETRENDED_FLUX'] = chunk['NORM_FLUX']  # skip detrending for tiny chunks

      df_detrended = pd.concat(chunks).reset_index(drop=True)

      time_array = df_detrended['TIME'].values
      flux_array = df_detrended['DETRENDED_FLUX'].values
      pos_corr1 = df_detrended['POS_CORR1'].values
      pos_corr2 = df_detrended['POS_CORR2'].values

      if verbose:
        print(f"Detrended flux std: {np.std(df_detrended['DETRENDED_FLUX']):.5f}")

      # Gaussian Process Detrending
      try:
        kernel = celerite2.terms.SHOTerm(sigma=filtered_std, rho=3.3, tau=10)
        gp = celerite2.GaussianProcess(kernel, mean=1.0)
        gp.compute(time_array, yerr=filtered_std)
        gp_trend = gp.predict(time_array, flux_array)
        gp_corrected_flux = df_detrended['DETRENDED_FLUX'] / gp_trend
      except Exception as gp_error:
            print(f"GP failed: {gp_error} — using detrended flux")
            gp_corrected_flux = df_detrended['DETRENDED_FLUX']

      if verbose:
        print(f"GP corrected flux std: {np.std(gp_corrected_flux):.5f}")
        print(f"GP corrected flux mean: {np.mean(gp_corrected_flux):.5f}")

      print(f"Processed: {filepath.split('/')[-1]} — {len(time_array)} points")
      return time_array, np.array(gp_corrected_flux), np.array(pos_corr1), np.array(pos_corr2)

  except Exception as e:
        print(f"ERROR processing {filepath.split('/')[-1]}: {e}")
        return None, None, None, None

In [ ]:
t, f, p1, p2 = process_sector(f'{DRIVE}/k2.fits', verbose=True)

In [ ]:
plt.figure(figsize=(14,4))
plt.subplot(1,2,1)
plt.suptitle('K2 Rolling Effect After Cleaning')
plt.subplot(1,2,1)
plt.scatter(t, f, color ='red', label = 'SAP FLUX', s = 0.5, alpha = 0.3)
plt.xlabel('Time')
plt.ylabel('SAP_FLUX')
plt.title('SAP_FLUX')
plt.subplot(1,2,2)
plt.scatter(t, p1, color ='blue', label = 'POS CORR1', s = 0.5, alpha = 0.3)
plt.xlabel('Time')
plt.ylabel('POS_CORR1')
plt.title('POS_CORR1')
plt.tight_layout()
plt.show()

# Phase 3 - Roll Correction

In [ ]:
plt.figure(figsize=(14,4))
plt.scatter(p1, f, color ='red', s = 0.5, alpha = 0.3)
plt.ylabel('Flux')
plt.xlabel('POS_CORR1')
plt.title('K2 Rolling Effect Correlation')
plt.tight_layout()
plt.show()

In [ ]:
boundaries = np.where(np.diff(t) > 0.1)[0] + 1
t_chunk = np.split(t, boundaries)
f_chunk = np.split(f, boundaries)
p1_chunk = np.split(p1, boundaries)
p2_chunk = np.split(p2, boundaries)

f_corrected_chunks = []
for i in range(len(f_chunk)):
  if len(f_chunk[i]) < 6:
    f_corrected_chunks.append(f_chunk[i])
    continue
  else:
    transit_mask = (f_chunk[i] > np.median(f_chunk[i]) - 2.5*np.std(f_chunk[i])) & (f_chunk[i] < np.median(f_chunk[i]) + 2.5*np.std(f_chunk[i]))
    design_matrix = np.column_stack((np.ones(len(p1_chunk[i])), p1_chunk[i], p2_chunk[i], p1_chunk[i]**2, p2_chunk[i]**2, p1_chunk[i]*p2_chunk[i], p1_chunk[i]**3, p2_chunk[i]**3, (p1_chunk[i]**2)*p2_chunk[i], p1_chunk[i]*(p2_chunk[i]**2)))
    coeff = np.linalg.lstsq(design_matrix[transit_mask], f_chunk[i][transit_mask], rcond=None)
    roll_trend = design_matrix @ coeff[0]
    f_roll_corr = f_chunk[i] / roll_trend
    f_corrected_chunks.append(f_roll_corr)

t_roll_corr = np.concatenate(t_chunk)
f_roll_corr = np.concatenate(f_corrected_chunks)
f_roll_corr = f_roll_corr / np.median(f_roll_corr)
p1_roll_corr = np.concatenate(p1_chunk)
p2_roll_corr = np.concatenate(p2_chunk)

print(np.std(f), np.std(f_roll_corr))
print(len(t_roll_corr), len(f_roll_corr), len(p1_roll_corr))

In [ ]:
print(len(t_roll_corr), len(f_roll_corr))
print(t_roll_corr[0], t_roll_corr[-1])
print(np.std(f_roll_corr))
print(np.any(np.isnan(f_roll_corr)))
print(np.any(np.isnan(t_roll_corr)))

* GP on binned centroid position — ATTEMPTED, ABANDONED  
* Result: negligible improvement (std 0.01790 -> 0.01778, 0.7%)  
* Issue: GP trend near zero at centroid edges caused flux amplification (max=1.313)  
* Decision: proceed with f_roll_corr from polynomial correction only  
Kept as scientific record of attempted method

The residual correlation is not coming from chunk-to-chunk inconsistency — it is intrinsic to the data itself. The relationship between flux and centroid position is genuinely nonlinear and non-stationary in a way that a 2D quadratic polynomial cannot fully capture. This is a known limitation of polynomial decorrelation for K2. We have reached the practical ceiling of what this approach can achieve on this dataset. Further polynomial refinement will give diminishing returns. The time series is clean, the baseline is flat, transits are already visible. This is sufficient for transit detection.

TPF Pixel-Level Decorrelation

In [ ]:
# Comment this cell on subsequent runs
from google.colab import files
tpf = lk.search_targetpixelfile('EPIC 246199087', mission='K2', campaign=12, exptime='short')
tpf = tpf.download()
tpf.to_fits('/content/trappist1_tpf.fits', overwrite=True)
files.download('/content/trappist1_tpf.fits')
print(tpf.shape)
tpf.plot()

In [ ]:
df_tpf = astropy.io.fits.open(f'{DRIVE}/trappist1_tpf.fits')
df_tpf.info()

In [ ]:
df_tpf[1].columns

In [ ]:
table = df_tpf[1].data
time = table["TIME"]
flux = table["FLUX"]
quality = table["QUALITY"]
flux_err = table["FLUX_ERR"]

In [ ]:
print(flux.shape)
quality = quality == 0
print(quality.shape)

In [ ]:
quality = table["QUALITY"] == 0
time_clean = time[quality]
flux_clean = flux[quality]
flux_err_clean = flux_err[quality]

print("Surviving cadences:", len(time_clean))

print("Time shape:", time_clean.shape)
print("Flux shape:", flux_clean.shape)
print(flux_err_clean.shape)

print(np.isnan(flux_clean).any())

In [ ]:
flux_clean = np.nan_to_num(flux_clean)
np.isnan(flux_clean).any()

In [ ]:
mean_img = np.nanmean(flux_clean, axis=0)
print(mean_img)

plt.figure(figsize=(7,6))
plt.imshow(mean_img, origin='lower')
plt.colorbar(label='Mean Flux')
plt.title('Mean TPF Flux Image')
plt.xlabel('Column (array index)')
plt.ylabel('Row (array index)')
plt.show()

In [ ]:
aperture_mask = np.zeros((9, 10), dtype=bool)
aperture_mask = mean_img > 100

print("Selected pixels:", aperture_mask.sum())

plt.figure(figsize=(7,6))
img = plt.imshow(mean_img, origin='lower', aspect='auto')
plt.contour(aperture_mask.astype(float),
            levels=[0.5],
            origin='lower',
            linewidths=2)

plt.colorbar(img, label='Mean Flux')
plt.title('Mean Image with Aperture Mask')
plt.xlabel('Column')
plt.ylabel('Row')
plt.show()

In [ ]:
pix = flux_clean[:, aperture_mask]

sap_flux = pix.sum(axis=1)

print(np.isnan(flux_clean[:, aperture_mask]).any())
sap_flux = sap_flux / np.median(sap_flux)
print("SAP flux shape:", sap_flux.shape)
print('Std of SAP flux: ', np.std(sap_flux))

plt.figure(figsize=(10, 4))
plt.plot(time_clean, sap_flux)
plt.axhline(1.0, linestyle='--', color='gray')
plt.xlabel('Time')
plt.ylabel('Normalised Flux')
plt.tight_layout()
plt.show()

In [ ]:

# Pixel extraction + design matrix
pix = flux_clean[:, aperture_mask]

sap_flux = pix.sum(axis=1)
sap_flux_norm = sap_flux / np.median(sap_flux)

X = pix / sap_flux[:, None]
X_design = np.column_stack([np.ones(len(sap_flux)), X])

# Chunk detection
dt = np.diff(time_clean)
boundary_idx = np.where(dt > 0.1)[0] + 1

time_chunks = np.split(time_clean, boundary_idx)
flux_chunks = np.split(sap_flux_norm, boundary_idx)
X_chunks = np.split(X_design, boundary_idx)

# Per-chunk loop
residual_chunks = []

for i, (tch, fch, Xch) in enumerate(zip(time_chunks, flux_chunks, X_chunks), start=1):

    npar = Xch.shape[1]

    if len(fch) < npar + 5:
        residual_chunks.append(fch.copy())
        print(f"Chunk {i}: skipped (too small), N={len(fch)}")
        continue

    med = np.median(fch)

    # CONDITION 1: hard flare ceiling
    hard_mask = (fch <= 1.5)

    if hard_mask.sum() <= npar:
        residual_chunks.append(fch.copy())
        print(f"Chunk {i}: skipped (hard clip removed too much), N={len(fch)}")
        continue

    # PASS 1: sigma on hard-clipped subset
    sigma1 = np.std(fch[hard_mask])

    if sigma1 == 0 or not np.isfinite(sigma1):
        residual_chunks.append(fch.copy())
        print(f"Chunk {i}: skipped (bad sigma1), N={len(fch)}")
        continue

    pass1_mask = np.abs(fch - med) <= 3.0 * sigma1

    # combine with hard clip
    mask1 = hard_mask & pass1_mask

    if mask1.sum() <= npar:
        residual_chunks.append(fch.copy())
        print(f"Chunk {i}: skipped (pass1 too aggressive), N={len(fch)}")
        continue

    # PASS 2: sigma using survivors of hard+pass1
    sigma2 = np.std(fch[mask1])

    if sigma2 == 0 or not np.isfinite(sigma2):
        residual_chunks.append(fch.copy())
        print(f"Chunk {i}: skipped (bad sigma2), N={len(fch)}")
        continue

    pass2_mask = np.abs(fch - med) <= 2.5 * sigma2

    # Final OOT mask = all three conditions
    oot_mask = hard_mask & pass1_mask & pass2_mask

    if oot_mask.sum() <= npar:
        residual_chunks.append(fch.copy())
        print(f"Chunk {i}: skipped (insufficient OOT rows), N={len(fch)}")
        continue

    # Fit on OOT only
    coef, _, _, _ = np.linalg.lstsq(
        Xch[oot_mask],
        fch[oot_mask],
        rcond=None
    )

    # Apply model to full chunk
    model = Xch @ coef

    # Subtract + recenter
    resid = fch - model + 1.0
    residual_chunks.append(resid)

    print(
        f"Chunk {i}: N={len(fch)}, "
        f"masked={len(fch)-oot_mask.sum()}, "
        f"std={np.std(resid):.6f}"
    )

# 4. Concatenate all chunks
flux_chunk_pld = np.concatenate(residual_chunks)

med = np.median(flux_chunk_pld)
sigma = np.std(flux_chunk_pld)

print("Median:", med)
print("Std:", sigma)

clip_mask = np.abs(flux_chunk_pld - med) <= 3.0 * sigma

flux_clipped = flux_chunk_pld[clip_mask]

print("Points kept:", len(flux_clipped), "/", len(flux_chunk_pld))
print("Clipped std:", np.std(flux_clipped))

print("\nFinal corrected shape:", flux_chunk_pld.shape)
print("Final std:", np.std(flux_chunk_pld))

print(np.linalg.matrix_rank(X_chunks[1]))
np.std(flux_chunks[1][flux_chunks[1] <= 1.5])

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(time_clean, flux_chunk_pld)
plt.axhline(1.0, linestyle='--', color='gray')
plt.ylim(0.95, 1.05)
plt.xlabel('Time')
plt.ylabel('Residual')
plt.title('Residual Plot')
plt.tight_layout()
plt.show()

In [ ]:
med = np.median(flux_chunk_pld)
sigma = np.std(flux_chunk_pld)

print("Median:", med)
print("Std:", sigma)

# 3-sigma symmetric mask about median
mask = np.abs(flux_chunk_pld - med) <= 3.0 * sigma

t_pld = time_clean[mask]
f_pld = flux_chunk_pld[mask]

print("Surviving cadences:", len(t_pld))
print("Original cadences:", len(time_clean))
print("Rejected cadences:", len(time_clean) - len(t_pld))
print("Final std:", np.std(f_pld))
print("Shapes:", t_pld.shape, f_pld.shape)
np.max(f_pld)

In [ ]:
# 10-minute bins (days)
bin_width = 10 / 1440.0

bins = int((t_pld.max() - t_pld.min()) / bin_width)

# Bin edges
bin_edges = np.linspace(t_pld.min(), t_pld.max(), bins + 1)

# Assign each cadence to a bin
bin_idx = np.digitize(t_pld, bin_edges)

t_bin_list = []
f_bin_list = []

for b in np.unique(bin_idx):
    m = (bin_idx == b)
    if np.any(m):
        t_bin_list.append(np.median(t_pld[m]))
        f_bin_list.append(np.median(f_pld[m]))

t_bin = np.array(t_bin_list)
f_bin = np.array(f_bin_list)

print("Unbinned points:", len(t_pld))
print("Binned points:", len(t_bin))
print("Unbinned std:", np.std(f_pld))
print("Binned std:", np.std(f_bin))
print("Shapes:", t_bin.shape, f_bin.shape)

# Phase 4 — CETRA Transit Search

In [ ]:
times = t_pld
fluxes = f_pld.copy()
flux_std = np.std(f_pld)
flux_err = np.full(len(t_pld), flux_std)

In [ ]:
lc = LightCurve(times, fluxes, flux_err)

In [ ]:
td = TransitDetector(lc)

In [ ]:
linear_result = td.linear_search()

In [ ]:
periodic_result = td.period_search(min_period=0.5, max_period=25.0, pgrid_oversample=1)

In [ ]:
print(periodic_result)
periodic_result.period_array
periodic_result.get_max_likelihood_parameters()
print(periodic_result.get_max_likelihood_parameters())
best = periodic_result.get_max_likelihood_parameters()
print(best.period)
print(best.t0)
print(best.duration)
print(best.depth)

In [ ]:
print(np.isnan(periodic_result.like_ratio_array).sum())
print(np.min(periodic_result.like_ratio_array))
print(np.max(periodic_result.like_ratio_array))
print(np.mean(periodic_result.like_ratio_array))

In [ ]:
planet = [1.51, 2.42, 4.05, 6.10, 9.21, 12.35, 18.77]
# Plot Periodogram
plt.figure(figsize = (14,4))
plt.plot(periodic_result.period_array, periodic_result.like_ratio_array)
for i in planet:
plt.axvline(i, color = 'red')
plt.axhline(160, color='gray', linestyle='--', alpha=1)
plt.xlabel('Period')
plt.ylabel('Likelihood Ratio')
plt.title('CETRA Periodogram')
plt.tight_layout()
plt.show()

In [ ]:
from scipy.signal import find_peaks
peak_indices, properties = find_peaks(periodic_result.like_ratio_array, height=160, distance=50, prominence=50)
peak_periods = periodic_result.period_array[peak_indices]
peak_likelihoods = periodic_result.like_ratio_array[peak_indices]
sort_idx1 = np.argsort(peak_likelihoods)
sort_idx1 = sort_idx1[::-1]
peak_periods_sorted = peak_periods[sort_idx1]
peak_likelihoods_sorted = peak_likelihoods[sort_idx1]
print(len(peak_indices))

for i in range(min(20, len(peak_periods_sorted))):
print(f"Period: {peak_periods_sorted[i]:.4f} days | Likelihood: {peak_likelihoods_sorted[i]:.1f}")

In [ ]:
planet = [1.51, 2.42, 4.05, 6.10, 9.21, 12.35, 18.77]
labels = ['Trappist-1b', 'Trappist-1c', 'Trappist-1d', 'Trappist-1e', 'Trappist-1f', 'Trappist-1g', 'Trappist-1h']

for i, detected in enumerate(peak_periods_sorted):
    for j, known in enumerate(planet):

        frac_diff = abs(detected - known) / known

        if frac_diff < 0.02:
            percent_diff = frac_diff * 100
            likelihood = peak_likelihoods_sorted[i]

            print(
                f"MATCH — {labels[j]} | "
                f"Known: {known:.4f} | "
                f"Detected: {detected:.4f} | "
                f"Diff: {percent_diff:.2f}% | "
                f"Likelihood: {likelihood:.2f}"
            )

        elif frac_diff < 0.05:
           percent_diff = frac_diff * 100
           likelihood = peak_likelihoods_sorted[i]

           print(
               f"NEAR MISS — {labels[j]} | "
               f"Known: {known:.4f} | "
               f"Detected: {detected:.4f} | "
               f"Diff: {percent_diff:.2f}% | "
               f"Likelihood: {likelihood:.2f}"
           )

In [ ]:
planet_info =[
    {'label': 'Trappist-1b', 'known_period': 1.5100, 'detected_period' : 1.4721, 'likelihood' : 1002.24, 'match_type' : 'Near Miss'},
    {'label': 'Trappist-1c', 'known_period': 2.4200, 'detected_period' : 2.4510, 'likelihood' : 636.91, 'match_type' : 'Match'},
    {'label': 'Trappist-1d', 'known_period': 4.0500, 'detected_period' : 3.9238, 'likelihood' : 689.13, 'match_type' : 'Near Miss'},
    {'label': 'Trappist-1e', 'known_period': 6.1000, 'detected_period' : 6.3704, 'likelihood' : 356.55, 'match_type' : 'Near Miss'},
    {'label': 'Trappist-1f', 'known_period': 9.2100, 'detected_period' : 9.0776, 'likelihood' : 356.31, 'match_type' : 'Match'},
    {'label': 'Trappist-1g', 'known_period': 12.3500, 'detected_period' : 12.8753, 'likelihood' : 558.76, 'match_type' : 'Near Miss'}
]

for planet in planet_info:
    print(planet['label'], planet['known_period'], planet['detected_period'], planet['likelihood'], planet['match_type'])

In [ ]:
results = []

for candidate in planet_info:
    detected_period = candidate["detected_period"]

    # closest period index
    idx = np.argmin(np.abs(periodic_result.period_array - detected_period))

    # t0
    t0_idx = periodic_result.t0_index_array[idx]
    t0 = periodic_result.linear_result.t0_array[t0_idx]

    # duration
    duration_idx = periodic_result.duration_index_array[idx]
    duration = periodic_result.linear_result.duration_array[duration_idx]

    # phase
    phase = (t_pld - t0) / detected_period
    phase = phase - np.floor(phase)
    phase[phase > 0.5] -= 1

    # binning
    half_width = max(0.05, 3 * duration / detected_period)

    bin_edges = np.linspace(-half_width, half_width, 401)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    digitized = np.digitize(phase, bin_edges)

    binned_flux = np.array([
        np.median(f_pld[digitized == i]) if np.any(digitized == i) else np.nan
        for i in range(1, len(bin_edges))
    ])

    #  t0 Refinement
    min_idx = np.nanargmin(binned_flux)

    phase_offset = bin_centers[min_idx]
    time_offset = phase_offset * detected_period

    t0_refined = t0 + time_offset

    # recompute phase with refined t0
    phase = (t_pld - t0_refined) / detected_period
    phase = phase - np.floor(phase)
    phase[phase > 0.5] -= 1

    # re-bin
    digitized = np.digitize(phase, bin_edges)

    binned_flux = np.array([
        np.median(f_pld[digitized == i]) if np.any(digitized == i) else np.nan
        for i in range(1, len(bin_edges))
    ])

    results.append({
        "period": detected_period,
        "idx": idx,
        "t0": t0_refined,
        "duration": duration,
        "phase": phase,
        "bin_centers": bin_centers,
        "binned_flux": binned_flux
    })

    print(f"{candidate['label']} | t0: {t0_refined:.4f} | duration: {duration*1440:.1f} min | half_width: {half_width:.3f}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.flatten()

for i, res in enumerate(results):
    ax = axes[i]

    ax.plot(res['bin_centers'], res['binned_flux'], linestyle='-', markersize=3)
    ax.axhline(1.0, linestyle='--')

    label = planet_info[i].get("label", f"Candidate {i+1}")
    period = planet_info[i].get("detected_period", None)
    match_type = planet_info[i].get("match_type", "N/A")
    likelihood = planet_info[i].get("likelihood", "N/A")

    ax.set_title(f"{label} | P={period:.4f} d | {match_type} | L={likelihood}")
    ax.set_xlabel("Phase")
    ax.set_ylabel("Normalised Flux")

plt.tight_layout()
plt.show()
plt.savefig(f'{DRIVE}/cetra_phase_folds.png', dpi=150, bbox_inches='tight')

In [ ]:
import os
import json

os.makedirs(DRIVE, exist_ok=True)

np.save(f"{DRIVE}/t_pld.npy", t_pld)
np.save(f"{DRIVE}/f_pld.npy", f_pld)
np.save(f"{DRIVE}/p1_roll_corr.npy", p1_roll_corr)
np.save(f"{DRIVE}/p2_roll_corr.npy", p2_roll_corr)

np.save(f"{DRIVE}/cetra_period_array.npy", periodic_result.period_array)
np.save(f"{DRIVE}/cetra_likelihood_array.npy", periodic_result.like_ratio_array)

for i, res in enumerate(results):
    np.save(f"{DRIVE}/cetra_bin_centers_{i}.npy", res['bin_centers'])
    np.save(f"{DRIVE}/cetra_binned_flux_{i}.npy", res['binned_flux'])

with open(f"{DRIVE}/cetra_planet_info.json", "w") as f:
    json.dump(planet_info, f, indent=4)

results_serializable = []
for i, res in enumerate(results):
    results_serializable.append({
        "period": float(res['period']),
        "t0": float(res['t0']),
        "duration": float(res['duration']),
        "idx": int(res['idx']),
        "label": planet_info[i]["label"]
    })

with open(f"{DRIVE}/cetra_results_params.json", "w") as f:
    json.dump(results_serializable, f, indent=4)

planet_periods = [1.51, 2.42, 4.05, 6.10, 9.21, 12.35, 18.77]
planet_labels = ['b', 'c', 'd', 'e', 'f', 'g', 'h']

plt.figure(figsize=(14, 4))
plt.plot(periodic_result.period_array, periodic_result.like_ratio_array)
plt.axhline(160, color='gray', linestyle='--', alpha=0.5)

for p, l in zip(planet_periods, planet_labels):
    plt.axvline(p, color='red', alpha=0.7)

plt.xlabel('Period (days)')
plt.ylabel('Likelihood Ratio')
plt.title('CETRA Periodogram — K2 Campaign 12')
plt.tight_layout()

plt.savefig(f"{DRIVE}/cetra_periodogram.png", dpi=150, bbox_inches='tight')
plt.close()

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.flatten()

for i, res in enumerate(results):
    ax = axes[i]

    ax.plot(res['bin_centers'], res['binned_flux'], marker='.', linewidth=1)
    ax.axhline(1.0, linestyle='--', color='gray')

    label = planet_info[i]['label']
    period = planet_info[i]['detected_period']
    match_type = planet_info[i]['match_type']
    likelihood = planet_info[i]['likelihood']

    ax.set_title(f"{label} | P={period:.4f} d | {match_type} | L={likelihood}")
    ax.set_xlabel('Phase')
    ax.set_ylabel('Normalised Flux')

for j in range(len(results), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(f"{DRIVE}/cetra_phase_folds.png", dpi=150, bbox_inches='tight')
plt.close()

print("Saved files:")
for f in os.listdir(DRIVE):
    print(f)

In [ ]:
import os
import json

t_pld = np.load(f"{DRIVE}/t_pld.npy")
f_pld = np.load(f"{DRIVE}/f_pld.npy")

p1_roll_corr = np.load(f"{DRIVE}/p1_roll_corr.npy")
p2_roll_corr = np.load(f"{DRIVE}/p2_roll_corr.npy")

periodic_result = np.load(f"{DRIVE}/cetra_period_array.npy")
cetra_likelihoods = np.load(f"{DRIVE}/cetra_likelihood_array.npy")

print("\nCore arrays loaded")
print("t_pld:", len(t_pld))
print("f_pld:", len(f_pld))

with open(f"{DRIVE}/cetra_planet_info.json", "r") as f:
    planet_info = json.load(f)

with open(f"{DRIVE}/cetra_results_params.json", "r") as f:
    cetra_results_params = json.load(f)

print("\nMetadata loaded")
print("Candidates:", len(planet_info))

results = []

for i in range(len(cetra_results_params)):
    try:
        bin_centers = np.load(f"{DRIVE}/cetra_bin_centers_{i}.npy")
        binned_flux = np.load(f"{DRIVE}/cetra_binned_flux_{i}.npy")

        results.append({
            "period": cetra_results_params[i]["period"],
            "t0": cetra_results_params[i]["t0"],
            "duration": cetra_results_params[i]["duration"],
            "idx": cetra_results_params[i]["idx"],
            "label": cetra_results_params[i]["label"],
            "bin_centers": bin_centers,
            "binned_flux": binned_flux
        })

    except FileNotFoundError:
        print(f"Missing phase fold files for index {i}")

print("\nResults reconstructed:", len(results))

print("\nSanity check:")
print("First candidate:", results[0]["label"])
print("First period:", results[0]["period"])
print("Sample phase:", results[0]["bin_centers"][:5])
print("Sample flux:", results[0]["binned_flux"][:5])

In [ ]:
planet_periods = [1.51, 2.42, 4.05, 6.10, 9.21, 12.35, 18.77]
planet_labels = ['b', 'c', 'd', 'e', 'f', 'g', 'h']

plt.figure(figsize=(14, 4))
plt.plot(periodic_result, cetra_likelihoods)
plt.axhline(160, color='gray', linestyle='--', alpha=0.5)
for p, l in zip(planet_periods, planet_labels):
    plt.axvline(p, color='red', alpha=0.7)
    # plt.text(p, 1700, l, color='red', fontsize=8, ha='left')
plt.xlabel('Period (days)')
plt.ylabel('Likelihood Ratio')
plt.title('CETRA Periodogram — K2 Campaign 12')
plt.tight_layout()
plt.savefig(f'{DRIVE}/cetra_periodogram.png', dpi=150, bbox_inches='tight')
plt.show()

# Save CETRA phase folds
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
axes = axes.flatten()
for i, res in enumerate(results):
    ax = axes[i]
    ax.plot(res['bin_centers'], res['binned_flux'])
    ax.axhline(1.0, linestyle='--', color='gray')
    label = planet_info[i]['label']
    period = planet_info[i]['detected_period']
    match_type = planet_info[i]['match_type']
    likelihood = planet_info[i]['likelihood']
    ax.set_title(f"{label} | P={period:.4f} d | {match_type} | L={likelihood}")
    ax.set_xlabel('Phase')
    ax.set_ylabel('Normalised Flux')
plt.tight_layout()
plt.savefig(f'{DRIVE}/cetra_phase_folds.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import json

# Load arrays
t_pld = np.load(f"{DRIVE}/t_pld.npy")
f_pld = np.load(f"{DRIVE}/f_pld.npy")
p1_roll_corr = np.load(f"{DRIVE}/p1_roll_corr.npy")
p2_roll_corr = np.load(f"{DRIVE}/p2_roll_corr.npy")
cetra_periods = np.load(f"{DRIVE}/cetra_period_array.npy")
cetra_likelihoods = np.load(f"{DRIVE}/cetra_likelihood_array.npy")

# Load json files
with open(f"{DRIVE}/cetra_planet_info.json", "r") as f:
    planet_info = json.load(f)

with open(f"{DRIVE}/cetra_results_params.json", "r") as f:
    cetra_results_params = json.load(f)

with open(f"{DRIVE}/tls_results.json", "r") as f:
    tls_results = json.load(f)

# Load phase fold arrays
results = []
for i in range(len(cetra_results_params)):
    results.append({
        **cetra_results_params[i],
        'bin_centers': np.load(f"{DRIVE}/cetra_bin_centers_{i}.npy"),
        'binned_flux': np.load(f"{DRIVE}/cetra_binned_flux_{i}.npy")
    })

print("All data loaded successfully")
print(f"t_pld: {len(t_pld)} points")
print(f"CETRA candidates: {len(planet_info)}")

* Detected 2 matches, 4 near misses, 1 non-detection out of 7 planets
* Roll systematic alias at 0.735 days dominated top of periodogram throughout
* Duration overestimation systematic identified — caused by noise floor exceeding per-point transit depth
* CETRA performance limited by 1.79% noise floor, not algorithm capability

# Phase 5 - Transit Least Square Algorithm

In [ ]:
# DUAL-BIN TLS
# Run 1 = 2-minute binned data for planet b
# Run 2 = 10-minute binned data for planets c-h

# BINNING FUNCTION
def time_bin(t, f, minutes):

    bin_width = minutes / 1440.0

    bins = int((t.max() - t.min()) / bin_width)
    edges = np.linspace(t.min(), t.max(), bins + 1)

    inds = np.digitize(t, edges)

    tb = []
    fb = []

    for b in np.unique(inds):
        m = inds == b
        if np.any(m):
            tb.append(np.median(t[m]))
            fb.append(np.median(f[m]))

    return np.array(tb), np.array(fb)


# BUILD BOTH BINNED DATASETS
# 2-minute for planet b
t_tls_b, f_tls_b = time_bin(t_pld, f_pld, 2)

# 10-minute for iterative search
t_tls, f_tls = time_bin(t_pld, f_pld, 10)

print("2-min points :", len(t_tls_b))
print("2-min std    :", np.std(f_tls_b))

print("10-min points:", len(t_tls))
print("10-min std   :", np.std(f_tls))

print("Unbinned std :", np.std(f_pld))

np.save(f"{DRIVE}/t_tls_b.npy", t_tls_b)
np.save(f"{DRIVE}/f_tls_b.npy", f_tls_b)
np.save(f"{DRIVE}/t_tls.npy", t_tls)
np.save(f"{DRIVE}/f_tls.npy", f_tls)

# HELPER: PHASE FOLD
def make_phase_fold(t, f, period, t0, duration):

    phase = (t - t0) / period
    phase = phase - np.floor(phase)
    phase[phase > 0.5] -= 1

    half_width = max(0.05, 2.5 * duration / period)

    edges = np.linspace(-half_width, half_width, 401)
    centers = 0.5 * (edges[:-1] + edges[1:])

    digitized = np.digitize(phase, edges)

    binned_flux = np.array([
        np.median(f[digitized == i]) if np.any(digitized == i) else np.nan
        for i in range(1, len(edges))
    ])

    # T0 refine
    min_idx = np.nanargmin(binned_flux)
    phase_offset = centers[min_idx]
    t0_refined = t0 + phase_offset * period

    # recompute
    phase = (t - t0_refined) / period
    phase = phase - np.floor(phase)
    phase[phase > 0.5] -= 1

    digitized = np.digitize(phase, edges)

    binned_flux = np.array([
        np.median(f[digitized == i]) if np.any(digitized == i) else np.nan
        for i in range(1, len(edges))
    ])

    return centers, binned_flux, half_width, t0_refined


# STORAGE
tls_results = []
phase_fold_data = []


# RUN 1 : PLANET b (2-minute binned)
print("\n" + "="*60)
print("RUN 1 : Planet b (2-minute)")
print("="*60)

model = transitleastsquares(t_tls_b, f_tls_b)

results = model.power(
    R_star=0.1179,
    M_star=0.0898,
    R_star_min=0.1079,
    R_star_max=0.1279,
    M_star_min=0.0798,
    M_star_max=0.0998,
    u=[0.4, 0.2],
    limb_dark="quadratic",
    period_min=1.48,
    period_max=1.90,
    oversampling_factor=3
)

print("SDE:", results.SDE)
print("Period:", results.period)

if np.isfinite(results.period) and results.SDE > 0:

    centers, bflux, half_width, t0_refined = make_phase_fold(
        t_tls_b, f_tls_b,
        results.period,
        results.T0,
        results.duration
    )

    tls_results.append({
        "run": 1,
        "iteration": 1,
        "label": "planet_b",
        "period": float(results.period),
        "t0": float(t0_refined),
        "duration": float(results.duration),
        "duration_min": float(results.duration * 1440),
        "depth": float(results.depth),
        "SDE": float(results.SDE),
        "snr": float(results.snr),
        "half_width": float(half_width)
    })

    phase_fold_data.append({
        "bin_centers": centers,
        "binned_flux": bflux
    })


# RUN 2 : PLANETS c-h (10-minute iterative)
print("\n" + "="*60)
print("RUN 2 : Planets c-h (10-minute iterative)")
print("="*60)

t_work = t_tls.copy()
f_work = f_tls.copy()

iteration = 1

while iteration <= 10:

    print(f"\nIteration {iteration}")
    print("Working points:", len(t_work))

    model = transitleastsquares(t_work, f_work)

    results = model.power(
        R_star=0.1179,
        M_star=0.0898,
        R_star_min=0.1079,
        R_star_max=0.1279,
        M_star_min=0.0798,
        M_star_max=0.0998,
        u=[0.4, 0.2],
        limb_dark="quadratic",
        period_min=2.3,
        period_max=20.0,
        oversampling_factor=1
    )

    print("SDE:", results.SDE)
    print("Period:", results.period)

    if results.SDE < 7 or not np.isfinite(results.period):
        print("Stopping.")
        break

    centers, bflux, half_width, t0_refined = make_phase_fold(
        t_work, f_work,
        results.period,
        results.T0,
        results.duration
    )

    tls_results.append({
        "run": 2,
        "iteration": iteration,
        "label": "planet_c_to_h",
        "period": float(results.period),
        "t0": float(t0_refined),
        "duration": float(results.duration),
        "duration_min": float(results.duration * 1440),
        "depth": float(results.depth),
        "SDE": float(results.SDE),
        "snr": float(results.snr),
        "half_width": float(half_width)
    })

    phase_fold_data.append({
        "bin_centers": centers,
        "binned_flux": bflux
    })

    # mask transits for next iteration
    mask = np.ones(len(t_work), dtype=bool)

    for tm in results.transit_times:
        in_transit = np.abs(t_work - tm) < (1.5 * results.duration/2)
        mask &= ~in_transit

    t_work = t_work[mask]
    f_work = f_work[mask]

    iteration += 1


# SAVE
with open(f"{DRIVE}/tls_results.json", "w") as f:
    json.dump(tls_results, f, indent=4)

print("\nTLS complete.")
print("Detections:", len(tls_results))


# PLOT
n = len(phase_fold_data)

if n > 0:

    ncols = 3
    nrows = int(np.ceil(n / ncols))

    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4*nrows))
    axes = axes.flatten()

    for i, (res, pf) in enumerate(zip(tls_results, phase_fold_data)):

        ax = axes[i]

        ax.plot(pf["bin_centers"], pf["binned_flux"], lw=1.4)
        ax.axhline(1.0, ls="--", alpha=0.6)

        ax.set_title(
            f"Run {res['run']} Iter {res['iteration']}\n"
            f"P={res['period']:.4f} d | SDE={res['SDE']:.1f}"
        )

        ax.set_xlabel("Phase")
        ax.set_ylabel("Flux")

    for j in range(n, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.savefig(f"{DRIVE}/tls_all_phase_folds.png", dpi=150)
    plt.show()


In [ ]:
from types import SimpleNamespace
from matplotlib.image import imread

# LOAD tls_results.json
with open(f"{DRIVE}/tls_results.json", "r") as f:
    tls_results = json.load(f)

t_tls_b = np.load(f"{DRIVE}/t_tls_b.npy")
f_tls_b = np.load(f"{DRIVE}/f_tls_b.npy")
t_tls = np.load(f"{DRIVE}/t_tls.npy")
f_tls = np.load(f"{DRIVE}/f_tls.npy")

print("Loaded detections:", len(tls_results))

for i, r in enumerate(tls_results, start=1):
    print(
        f"{i}. Run {r['run']} Iter {r['iteration']} | "
        f"P={r['period']:.6f} d | "
        f"SDE={r['SDE']:.2f} | "
        f"Label={r['label']}"
    )

# convert entries to objects like TLS result
tls_objects = []

for r in tls_results:
    obj = SimpleNamespace(**r)
    tls_objects.append(obj)

# LOAD AND DISPLAY SAVED PLOT
img = imread(f"{DRIVE}/tls_all_phase_folds.png")

plt.figure(figsize=(16, 10))
plt.imshow(img)
plt.axis("off")
plt.title("Saved Main TLS Phase Fold Plot")
plt.show()

In [ ]:
# INSPECT 16.1826-DAY DETECTION USING UNBINNED PLD DATA

target_period = 16.1826

# find matching detection
match = None
for r in tls_results:
    if np.isfinite(r["period"]) and abs(r["period"] - target_period) < 0.05:
        match = r
        break

if match is None:
    print("No matching detection found near 16.1826 d")
else:

    period = 16.1826
    t0 = match["t0"]

    print("Using period:", period)
    print("Using t0:", t0)

    # expected transits
    transit_times = [t0 + n * period for n in range(5)]

    # ±2 hours
    half_window = 2 / 24.0   # days

    fig, axes = plt.subplots(5, 1, figsize=(10, 14), sharex=False)

    for i, tc in enumerate(transit_times):

        ax = axes[i]

        mask = np.abs(t_pld - tc) <= half_window

        tw = t_pld[mask]
        fw = f_pld[mask]

        if len(tw) == 0:
            ax.text(0.5, 0.5, "No data", ha="center", va="center")
            ax.set_title(f"Transit {i} | Tc = {tc:.5f}")
            continue

        # convert x-axis to hours from mid-transit
        x = (tw - tc) * 24.0

        ax.plot(x, fw, ms=2, alpha=0.8)
        ax.axvline(0, ls="--", alpha=0.6)
        ax.axhline(1.0, ls="--", alpha=0.5)

        ax.set_title(f"Transit {i} | Tc = {tc:.5f}")
        ax.set_ylabel("Flux")
        ax.set_xlim(-2, 2)

    axes[-1].set_xlabel("Hours from predicted mid-transit")

    plt.tight_layout()
    plt.show()

For Period 16.1826:  
Transit 0 — Tc=2910.18: No dip at hour 0. Noisy baseline, dip visible around hour -0.8 but not at the predicted time.  
Transit 1 — Tc=2926.36: Baseline is depressed across the entire window at ~0.98 — this is a long-duration systematic, not a transit. The flux recovers after hour 0, suggesting this window falls inside a roll systematic episode.  
Transit 2 — Tc=2942.55: Large data gap right at hour 0 — the predicted transit falls in the missing data region. No information here.  
Transit 3 — Tc=2958.73: Falls in the inter-chunk gap (BKJD ~2955-2960). No transit visible, just noise.  
Transit 4 — Tc=2974.91: No dip at hour 0. Flat noisy baseline.  
Verdict: Not a real planet.  
Confirmed FALSE ALARM.

In [ ]:
def make_phase_fold(t, f, period, t0, duration):

    phase = (t - t0) / period
    phase = phase - np.floor(phase)
    phase[phase > 0.5] -= 1

    half_width = max(0.05, 2.5 * duration / period)

    edges = np.linspace(-half_width, half_width, 401)
    centers = 0.5 * (edges[:-1] + edges[1:])

    digitized = np.digitize(phase, edges)

    binned_flux = np.array([
        np.median(f[digitized == i]) if np.any(digitized == i) else np.nan
        for i in range(1, len(edges))
    ])

    # refine T0 from minimum bin
    if np.all(np.isnan(binned_flux)):
        return centers, binned_flux, half_width, t0

    min_idx = np.nanargmin(binned_flux)
    phase_offset = centers[min_idx]
    t0_refined = t0 + phase_offset * period

    # recompute with refined T0
    phase = (t - t0_refined) / period
    phase = phase - np.floor(phase)
    phase[phase > 0.5] -= 1

    digitized = np.digitize(phase, edges)

    binned_flux = np.array([
        np.median(f[digitized == i]) if np.any(digitized == i) else np.nan
        for i in range(1, len(edges))
    ])

    return centers, binned_flux, half_width, t0_refined

In [ ]:
# TARGETED TLS SEARCHES : PLANETS e, f, g, h

print("\n" + "="*70)
print("TARGETED TLS SEARCHES : e, f, g, h")
print("="*70)

tls_results_targetted = []
phase_fold_data = []

targets = [
    ("planet_e", 5.185, 7.015),
    ("planet_f", 7.826, 10.588),
    ("planet_g", 10.500, 14.206),
    ("planet_h", 15.952, 21.582),
]

for i, (label, pmin, pmax) in enumerate(targets, start=1):

    print("\n" + "-"*60)
    print(label)
    print(f"period_min = {pmin}")
    print(f"period_max = {pmax}")
    print("-"*60)

    model = transitleastsquares(t_tls, f_tls)

    results = model.power(
        R_star=0.1179,
        M_star=0.0898,
        R_star_min=0.1079,
        R_star_max=0.1279,
        M_star_min=0.0798,
        M_star_max=0.0998,
        u=[0.4, 0.2],
        limb_dark="quadratic",
        period_min=pmin,
        period_max=pmax,
        oversampling_factor=3
    )

    print("SDE:", results.SDE)
    print("Period:", results.period)
    print("T0:", results.T0)
    print("Duration (min):", results.duration * 1440)
    print("Depth:", results.depth)

    if np.isfinite(results.period):

        centers, bflux, half_width, t0_refined = make_phase_fold(
            t_tls,
            f_tls,
            results.period,
            results.T0,
            results.duration
        )

        tls_results_targetted.append({
            "run": i,
            "iteration": 1,
            "label": label,
            "period": float(results.period),
            "t0": float(t0_refined),
            "duration": float(results.duration),
            "duration_min": float(results.duration * 1440),
            "depth": float(results.depth),
            "SDE": float(results.SDE),
            "snr": float(results.snr),
            "half_width": float(half_width)
        })

        phase_fold_data.append({
            "bin_centers": centers,
            "binned_flux": bflux
        })


# SAVE RESULTS
with open(f"{DRIVE}/tls_targeted_efgh_results.json", "w") as f:
    json.dump(tls_results_targetted, f, indent=4)

print("\nCompleted targeted runs.")
print("Detections stored:", len(tls_results_targetted))


# PLOT ALL FOUR PHASE FOLDS
n = len(phase_fold_data)

if n > 0:

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for i, (res, pf) in enumerate(zip(tls_results_targetted, phase_fold_data)):

        ax = axes[i]

        ax.plot(pf["bin_centers"], pf["binned_flux"], lw=1.5)
        ax.axhline(1.0, ls="--", alpha=0.6)
        ax.axvline(0.0, ls="--", alpha=0.4)

        ax.set_title(
            f"{res['label']}\n"
            f"P={res['period']:.4f} d | "
            f"SDE={res['SDE']:.2f}"
        )

        ax.set_xlabel("Phase")
        ax.set_ylabel("Flux")

    for j in range(n, 4):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.savefig(
        f"{DRIVE}/tls_targeted_efgh_phasefolds.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.show()

TLS was run in two configurations on 10-minute binned PLD data (10,396 points, std=0.00707):  
* Run 1 — Targeted planet b search (2-minute binned data, period 1.48-1.90 days):
Planet b detected cleanly at P=1.5109 days, SDE=37.4, offset 0.08%. Clean narrow dip confirmed in phase fold.  

* Run 2 — Iterative search period 2.3-20.0 days:  
  * Confirmed detections:  
    * Planet c (2.4219 d, very precise match)  
    * Planet d (3.9279 d, strong signal)
  * Strong new candidates:  
    * Planets e, f, g (good SDE, moderate offsets)
  * Marginal candidate:  
    * Planet h (low SDE, high offset)
  * Duplicate signal:  
    * 3.9263 d (repeat of planet d)
  * False alarms:  
    * Signals around 15–16 days




# Phase 6 - Glance Diagnostic Suite

### For TLS

In [ ]:
# ODD-EVEN TEST FOR ALL 7 TRAPPIST-1 PLANETS

# PLANET PARAMETERS
# duration in days
planets = [
    {"label": "planet_b", "period": 1.5109,  "duration": 30.4 / 1440},
    {"label": "planet_c", "period": 2.4219,  "duration": 36.0 / 1440},
    {"label": "planet_d", "period": 3.9279,  "duration": 42.0 / 1440},
    {"label": "planet_e", "period": 5.8910,  "duration": 48.0 / 1440},
    {"label": "planet_f", "period": 9.8162,  "duration": 55.0 / 1440},
    {"label": "planet_g", "period": 11.6624, "duration": 62.0 / 1440},
    {"label": "planet_h", "period": 21.3586, "duration": 76.0 / 1440},
]

# MATCH t0 FROM tls_results BY CLOSEST PERIOD
print("=" * 60)
print("MATCHING PLANETS TO tls_results")
print("=" * 60)

for planet in planets:

    best = min(
        tls_results,
        key=lambda r: abs(r["period"] - planet["period"])
    )

    planet["t0"] = best["t0"]
    planet["matched_period"] = best["period"]

    print(
        f"{planet['label']:8s} | target={planet['period']:.4f} d "
        f"| matched={best['period']:.4f} d "
        f"| t0={best['t0']:.6f}"
    )

# BINNING FUNCTION
def odd_even_fold(times, fluxes, transit_times, duration, period):

    phase_all = []
    flux_all = []

    for tc in transit_times:

        mask = np.abs(times - tc) <= (2 * duration)

        if np.any(mask):
            ph = (times[mask] - tc) / period
            phase_all.extend(ph)
            flux_all.extend(fluxes[mask])

    phase_all = np.array(phase_all)
    flux_all = np.array(flux_all)

    half_width = max(0.05, 2.5 * duration / period)

    edges = np.linspace(-half_width, half_width, 401)
    centers = 0.5 * (edges[:-1] + edges[1:])

    inds = np.digitize(phase_all, edges)

    binned = np.array([
        np.median(flux_all[inds == i]) if np.any(inds == i) else np.nan
        for i in range(1, len(edges))
    ])

    return centers, binned

# PLOT GRID
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

for k, planet in enumerate(planets):

    ax = axes[k]

    label = planet["label"]
    period = planet["period"]
    duration = planet["duration"]
    t0 = planet["t0"]

    # Generate all transit numbers in baseline
    n_start = int(np.floor((t_pld.min() - t0) / period))
    n_end   = int(np.ceil((t_pld.max() - t0) / period))

    n_vals = np.arange(n_start, n_end + 1)

    t_transits = t0 + n_vals * period

    valid = (t_transits >= t_pld.min()) & (t_transits <= t_pld.max())

    n_vals = n_vals[valid]
    t_transits = t_transits[valid]

    # Split odd/even
    even_times = t_transits[n_vals % 2 == 0]
    odd_times  = t_transits[n_vals % 2 == 1]

    # Fold separately
    c_even, f_even = odd_even_fold(
        t_pld, f_pld, even_times, duration, period
    )

    c_odd, f_odd = odd_even_fold(
        t_pld, f_pld, odd_times, duration, period
    )

    # Plot
    ax.plot(c_even, f_even, lw=1.5, label="Even")
    ax.plot(c_odd,  f_odd,  lw=1.5, label="Odd")

    ax.axhline(1.0, ls="--", alpha=0.5)
    ax.axvline(0.0, ls="--", alpha=0.4)

    ax.set_title(
        f"{label}\n"
        f"P={period:.4f} d"
    )

    ax.set_xlabel("Phase")
    ax.set_ylabel("Flux")
    ax.legend(fontsize=8)

for j in range(len(planets), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# SECONDARY ECLIPSE TEST FOR ALL 7 TRAPPIST-1 PLANETS

# PLANET PARAMETERS (TLS periods)
planets = [
    {"label": "planet_b", "period": 1.5109},
    {"label": "planet_c", "period": 2.4219},
    {"label": "planet_d", "period": 3.9279},
    {"label": "planet_e", "period": 5.8910},
    {"label": "planet_f", "period": 9.8162},
    {"label": "planet_g", "period": 11.6624},
    {"label": "planet_h", "period": 21.3586},
]

# MATCH t0 FROM tls_results BY CLOSEST PERIOD
print("=" * 70)
print("MATCHING TLS PLANETS")
print("=" * 70)

for planet in planets:

    best = min(
        tls_results,
        key=lambda r: abs(r["period"] - planet["period"])
    )

    planet["t0"] = best["t0"]
    planet["matched_period"] = best["period"]

    print(
        f"{planet['label']:8s} | target={planet['period']:.4f} d "
        f"| matched={best['period']:.4f} d "
        f"| t0={best['t0']:.6f}"
    )

# FULL PHASE FOLD FUNCTION
def full_phase_fold(times, fluxes, period, t0):

    phase = (times - t0) / period
    phase = phase - np.floor(phase)
    phase[phase > 0.5] -= 1

    half_width = 0.5

    edges = np.linspace(-half_width, half_width, 401)
    centers = 0.5 * (edges[:-1] + edges[1:])

    inds = np.digitize(phase, edges)

    binned = np.array([
        np.median(fluxes[inds == i]) if np.any(inds == i) else np.nan
        for i in range(1, len(edges))
    ])

    return centers, binned

# PLOT GRID
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

for k, planet in enumerate(planets):

    ax = axes[k]

    label  = planet["label"]
    period = planet["period"]
    t0     = planet["t0"]

    centers, binned = full_phase_fold(
        t_pld, f_pld, period, t0
    )

    ax.plot(centers, binned, lw=1.4)

    # reference lines
    ax.axhline(1.0, ls="--", alpha=0.5)
    ax.axvline(0.0, ls="--", alpha=0.6)     # primary transit
    ax.axvline(-0.5, ls=":", alpha=0.4)     # secondary eclipse region
    ax.axvline(0.5, ls=":", alpha=0.4)

    ax.set_title(
        f"{label}\n"
        f"P={period:.4f} d"
    )

    ax.set_xlabel("Phase")
    ax.set_ylabel("Flux")
    ax.set_xlim(-0.5, 0.5)

# hide unused panel
for j in range(len(planets), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# CENTROID PHASE-FOLD TEST

# BUILD ROW/COL INDEX ARRAYS (9 x 10)
rows = np.arange(9)
cols = np.arange(10)

row_grid, col_grid = np.meshgrid(rows, cols, indexing="ij")

# MASK TO APERTURE PIXELS ONLY
row_ap = row_grid[aperture_mask]
col_ap = col_grid[aperture_mask]

pix_flux = flux_clean[:, aperture_mask]

# FLUX-WEIGHTED CENTROIDS
den = np.sum(pix_flux, axis=1)

centr_row = np.sum(pix_flux * row_ap[None, :], axis=1) / den
centr_col = np.sum(pix_flux * col_ap[None, :], axis=1) / den

# NORMALISE
centr_row_norm = centr_row - np.median(centr_row)
centr_col_norm = centr_col - np.median(centr_col)

# TLS PLANETS
planets = [
    {"label": "planet_b", "period": 1.5109},
    {"label": "planet_c", "period": 2.4219},
    {"label": "planet_d", "period": 3.9279},
    {"label": "planet_e", "period": 5.8910},
    {"label": "planet_f", "period": 9.8162},
    {"label": "planet_g", "period": 11.6624},
    {"label": "planet_h", "period": 21.3586},
]

# match t0 from tls_results
for planet in planets:
    best = min(
        tls_results,
        key=lambda r: abs(r["period"] - planet["period"])
    )
    planet["t0"] = best["t0"]

# PHASE FOLD FUNCTION
def centroid_fold(times, values, period, t0):

    phase = (times - t0) / period
    phase = phase - np.floor(phase)
    phase[phase > 0.5] -= 1

    edges = np.linspace(-0.1, 0.1, 401)
    centers = 0.5 * (edges[:-1] + edges[1:])

    inds = np.digitize(phase, edges)

    binned = np.array([
        np.median(values[inds == i]) if np.any(inds == i) else np.nan
        for i in range(1, len(edges))
    ])

    return centers, binned

# PLOT
fig, axes = plt.subplots(7, 2, figsize=(12, 24))

for i, planet in enumerate(planets):

    period = planet["period"]
    t0 = planet["t0"]

    c1, rbin = centroid_fold(time_clean, centr_row_norm, period, t0)
    c2, cbin = centroid_fold(time_clean, centr_col_norm, period, t0)

    ax1 = axes[i, 0]
    ax2 = axes[i, 1]

    ax1.plot(c1, rbin, lw=1.3)
    ax2.plot(c2, cbin, lw=1.3)

    ax1.axhline(0, ls="--", alpha=0.5)
    ax2.axhline(0, ls="--", alpha=0.5)

    ax1.axvline(0, ls="--", alpha=0.4)
    ax2.axvline(0, ls="--", alpha=0.4)

    ax1.set_title(f"{planet['label']} Row")
    ax2.set_title(f"{planet['label']} Col")

    ax1.set_ylabel("Centroid shift")
    ax2.set_ylabel("Centroid shift")

    ax1.set_xlabel("Phase")
    ax2.set_xlabel("Phase")

plt.tight_layout()
plt.show()

In [ ]:
# INDIVIDUAL TRANSIT CONSISTENCY TEST

rng = np.random.default_rng(42)

planets = [
    {"label": "planet_b", "period": 1.5109,  "duration": 30.4/1440},
    {"label": "planet_c", "period": 2.4219,  "duration": 36.0/1440},
    {"label": "planet_d", "period": 3.9279,  "duration": 42.0/1440},
    {"label": "planet_e", "period": 5.8910,  "duration": 48.0/1440},
    {"label": "planet_f", "period": 9.8162,  "duration": 55.0/1440},
    {"label": "planet_g", "period": 11.6624, "duration": 62.0/1440},
    {"label": "planet_h", "period": 21.3586, "duration": 76.0/1440},
]

# match t0 from tls_results
for p in planets:
    best = min(
        tls_results,
        key=lambda r: abs(r["period"] - p["period"])
    )
    p["t0"] = best["t0"]

def stacked_fold(times, fluxes, transit_times, duration, period):

    phase_all = []
    flux_all = []

    for tc in transit_times:

        mask = np.abs(times - tc) <= (2 * duration)

        if np.any(mask):
            ph = (times[mask] - tc) / period
            phase_all.extend(ph)
            flux_all.extend(fluxes[mask])

    phase_all = np.array(phase_all)
    flux_all = np.array(flux_all)

    half_width = max(0.05, 2.5 * duration / period)

    edges = np.linspace(-half_width, half_width, 401)
    centers = 0.5 * (edges[:-1] + edges[1:])

    inds = np.digitize(phase_all, edges)

    binned = np.array([
        np.median(flux_all[inds == i]) if np.any(inds == i) else np.nan
        for i in range(1, len(edges))
    ])

    return centers, binned, half_width

fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

for k, p in enumerate(planets):

    ax = axes[k]

    period = p["period"]
    t0 = p["t0"]
    duration = p["duration"]

    n_start = int(np.floor((t_pld.min() - t0) / period))
    n_end   = int(np.ceil((t_pld.max() - t0) / period))

    n_vals = np.arange(n_start, n_end + 1)
    t_transits = t0 + n_vals * period

    valid = (t_transits >= t_pld.min()) & (t_transits <= t_pld.max())
    t_transits = t_transits[valid]

    # sample limit for crowded planets
    if p["label"] in ["planet_b", "planet_c"] and len(t_transits) > 20:
        idx = rng.choice(len(t_transits), 20, replace=False)
        t_plot = np.sort(t_transits[idx])
    else:
        t_plot = t_transits

    # plot individual transits
    for tc in t_plot:

        mask = np.abs(t_pld - tc) <= (2 * duration)

        if np.any(mask):
            ph = (t_pld[mask] - tc) / period
            fw = f_pld[mask]

            ax.plot(ph, fw, color="gray", alpha=0.3, lw=0.8)

    # stacked median
    c, bf, hw = stacked_fold(
        t_pld, f_pld, t_transits, duration, period
    )

    ax.plot(c, bf, lw=2.0, color="blue")

    ax.axhline(1.0, ls="--", alpha=0.5)
    ax.axvline(0.0, ls="--", alpha=0.4)

    ax.set_xlim(-hw, hw)
    ax.set_title(f"{p['label']}\nP={period:.4f} d")
    ax.set_xlabel("Phase")
    ax.set_ylabel("Flux")

for j in range(len(planets), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# CONTAMINATION / APERTURE CHECK
# Applies equally to TLS and CETRA because both use same TPF data

# Required inputs:
# flux_clean      shape = (ntime, 9, 10)
# aperture_mask   shape = (9, 10) boolean

# MEAN TPF IMAGE WITH APERTURE OVERLAY
mean_img = np.nanmean(flux_clean, axis=0)

plt.figure(figsize=(10, 6))

im = plt.imshow(
    mean_img,
    origin="lower",
    aspect="auto"
)

plt.colorbar(im, label="Mean Flux")

# aperture contour
plt.contour(
    aperture_mask.astype(float),
    levels=[0.5],
    linewidths=2
)

# annotate bright pixels outside aperture
outside = ~aperture_mask

outside_vals = mean_img[outside]
threshold = np.nanpercentile(outside_vals, 95)

rows, cols = np.where(outside)

for r, c in zip(rows, cols):

    if mean_img[r, c] >= threshold:
        plt.text(
            c, r,
            f"{mean_img[r,c]:.0f}",
            ha="center",
            va="center",
            fontsize=8
        )

plt.title("Mean TPF Image + Aperture Contour")
plt.xlabel("Column")
plt.ylabel("Row")
plt.show()

# CONTAMINATION FRACTION
total_flux = np.nansum(mean_img)

inside_flux = np.nansum(mean_img[aperture_mask])

outside_flux = np.nansum(mean_img[~aperture_mask])

contam_frac = outside_flux / total_flux

print("=" * 60)
print("CONTAMINATION FRACTION")
print("=" * 60)
print("Total flux         :", total_flux)
print("Inside aperture    :", inside_flux)
print("Outside aperture   :", outside_flux)
print("Outside fraction   :", contam_frac)
print("Percent outside    :", 100 * contam_frac, "%")

# CHECK COLUMN 0 BACKGROUND FEATURE
# all pixels in column 0
col0_flux = np.nansum(mean_img[:, 0])

# brightest pixel in frame (proxy for source peak)
peak_flux = np.nanmax(mean_img)

ratio = col0_flux / peak_flux

print("\n" + "=" * 60)
print("COLUMN 0 BACKGROUND FEATURE")
print("=" * 60)
print("Column 0 total flux :", col0_flux)
print("Peak pixel flux     :", peak_flux)
print("Relative ratio      :", ratio)
print("Percent of peak     :", 100 * ratio, "%")

if ratio < 0.01:
    print("Verdict: Negligible (<1% of peak flux)")
else:
    print("Verdict: Non-negligible, inspect further")

 Diagnostic 6 is about the aperture and pixel data, which is independent of the detection algorithm. CETRA and TLS both operate on the same light curve extracted from the same aperture. Hence Diagnostic 6 isdone only once.

### For CETRA

In [ ]:
# ODD-EVEN TEST FOR ALL 7 TRAPPIST-1 PLANETS

planets = [
    {"label": "planet_b", "period": 1.4721,  "t0": 2906.0012, "duration": 30.4 / 1440},
    {"label": "planet_c", "period": 2.4510,  "t0": 2906.5254, "duration": 36.0 / 1440},
    {"label": "planet_d", "period": 3.9238,  "t0": 2906.2632, "duration": 42.0 / 1440},
    {"label": "planet_e", "period": 6.3704,  "t0": 2905.5554, "duration": 48.0 / 1440},
    {"label": "planet_f", "period": 9.0776,  "t0": 2911.8898, "duration": 55.0 / 1440},
    {"label": "planet_g", "period": 12.8753, "t0": 2907.6576, "duration": 62.0 / 1440},
]

# BINNING FUNCTION
def odd_even_fold(times, fluxes, transit_times, duration, period):

    phase_all = []
    flux_all = []

    for tc in transit_times:

        mask = np.abs(times - tc) <= (2 * duration)

        if np.any(mask):
            ph = (times[mask] - tc) / period
            phase_all.extend(ph)
            flux_all.extend(fluxes[mask])

    phase_all = np.array(phase_all)
    flux_all = np.array(flux_all)

    half_width = max(0.05, 2.5 * duration / period)

    edges = np.linspace(-half_width, half_width, 401)
    centers = 0.5 * (edges[:-1] + edges[1:])

    inds = np.digitize(phase_all, edges)

    binned = np.array([
        np.median(flux_all[inds == i]) if np.any(inds == i) else np.nan
        for i in range(1, len(edges))
    ])

    return centers, binned

# PLOT GRID
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

for k, planet in enumerate(planets):

    ax = axes[k]

    label = planet["label"]
    period = planet["period"]
    duration = planet["duration"]
    t0 = planet["t0"]

    # Generate all transit numbers in baseline
    n_start = int(np.floor((t_pld.min() - t0) / period))
    n_end   = int(np.ceil((t_pld.max() - t0) / period))

    n_vals = np.arange(n_start, n_end + 1)

    t_transits = t0 + n_vals * period

    valid = (t_transits >= t_pld.min()) & (t_transits <= t_pld.max())

    n_vals = n_vals[valid]
    t_transits = t_transits[valid]

    # Split odd/even
    even_times = t_transits[n_vals % 2 == 0]
    odd_times  = t_transits[n_vals % 2 == 1]

    # Fold separately
    c_even, f_even = odd_even_fold(
        t_pld, f_pld, even_times, duration, period
    )

    c_odd, f_odd = odd_even_fold(
        t_pld, f_pld, odd_times, duration, period
    )

    # Plot
    ax.plot(c_even, f_even, lw=1.5, label="Even")
    ax.plot(c_odd,  f_odd,  lw=1.5, label="Odd")

    ax.axhline(1.0, ls="--", alpha=0.5)
    ax.axvline(0.0, ls="--", alpha=0.4)

    ax.set_title(
        f"{label}\n"
        f"P={period:.4f} d"
    )

    ax.set_xlabel("Phase")
    ax.set_ylabel("Flux")
    ax.legend(fontsize=8)

for j in range(len(planets), len(axes)):
    axes[j].set_visible(False)

# Final empty panel for h (no CETRA detection)
axes[6].text(
    0.5, 0.5,
    "planet_h\nNo CETRA detection",
    ha="center",
    va="center",
    fontsize=11
)

plt.tight_layout()
plt.show()

In [ ]:
# SECONDARY ECLIPSE TEST FOR ALL 7 TRAPPIST-1 PLANETS

# PLANET PARAMETERS (CETRA)
planets = [
    {"label": "planet_b", "period": 1.4721,  "t0": 2906.0012},
    {"label": "planet_c", "period": 2.4510,  "t0": 2906.5254},
    {"label": "planet_d", "period": 3.9238,  "t0": 2906.2632},
    {"label": "planet_e", "period": 6.3704,  "t0": 2905.5554},
    {"label": "planet_f", "period": 9.0776,  "t0": 2911.8898},
    {"label": "planet_g", "period": 12.8753, "t0": 2907.6576},
]

print("=" * 70)
print("CETRA SECONDARY ECLIPSE TEST")
print("=" * 70)

for planet in planets:
    print(
        f"{planet['label']:8s} | "
        f"P={planet['period']:.4f} d | "
        f"t0={planet['t0']:.6f}"
    )

# FULL PHASE FOLD FUNCTION
def full_phase_fold(times, fluxes, period, t0):

    phase = (times - t0) / period
    phase = phase - np.floor(phase)
    phase[phase > 0.5] -= 1

    half_width = 0.5

    edges = np.linspace(-half_width, half_width, 401)
    centers = 0.5 * (edges[:-1] + edges[1:])

    inds = np.digitize(phase, edges)

    binned = np.array([
        np.median(fluxes[inds == i]) if np.any(inds == i) else np.nan
        for i in range(1, len(edges))
    ])

    return centers, binned

# PLOT GRID
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

for k, planet in enumerate(planets):

    ax = axes[k]

    label  = planet["label"]
    period = planet["period"]
    t0     = planet["t0"]

    centers, binned = full_phase_fold(
        t_pld, f_pld, period, t0
    )

    ax.plot(centers, binned, lw=1.4)

    # reference lines
    ax.axhline(1.0, ls="--", alpha=0.5)
    ax.axvline(0.0, ls="--", alpha=0.6)     # primary transit
    ax.axvline(-0.5, ls=":", alpha=0.4)     # secondary eclipse region
    ax.axvline(0.5, ls=":", alpha=0.4)

    ax.set_title(
        f"{label}\n"
        f"P={period:.4f} d"
    )

    ax.set_xlabel("Phase")
    ax.set_ylabel("Flux")
    ax.set_xlim(-0.5, 0.5)

# hide unused panels
for j in range(len(planets), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# CENTROID PHASE-FOLD TEST (CETRA PERIODS / T0 VALUES)

# BUILD ROW/COL INDEX ARRAYS
rows = np.arange(9)
cols = np.arange(10)

row_grid, col_grid = np.meshgrid(rows, cols, indexing="ij")

# APERTURE PIXELS ONLY
row_ap = row_grid[aperture_mask]
col_ap = col_grid[aperture_mask]

pix_flux = flux_clean[:, aperture_mask]

# CENTROIDS
den = np.sum(pix_flux, axis=1)

centr_row = np.sum(pix_flux * row_ap[None, :], axis=1) / den
centr_col = np.sum(pix_flux * col_ap[None, :], axis=1) / den

# NORMALISE
centr_row_norm = centr_row - np.median(centr_row)
centr_col_norm = centr_col - np.median(centr_col)

# CETRA PLANETS
planets = [
    {"label": "planet_b", "period": 1.4721,  "t0": 2906.0012},
    {"label": "planet_c", "period": 2.4510,  "t0": 2906.5254},
    {"label": "planet_d", "period": 3.9238,  "t0": 2906.2632},
    {"label": "planet_e", "period": 6.3704,  "t0": 2905.5554},
    {"label": "planet_f", "period": 9.0776,  "t0": 2911.8898},
    {"label": "planet_g", "period": 12.8753, "t0": 2907.6576},
]

# PHASE FOLD FUNCTION
def centroid_fold(times, values, period, t0):

    phase = (times - t0) / period
    phase = phase - np.floor(phase)
    phase[phase > 0.5] -= 1

    edges = np.linspace(-0.1, 0.1, 401)
    centers = 0.5 * (edges[:-1] + edges[1:])

    inds = np.digitize(phase, edges)

    binned = np.array([
        np.median(values[inds == i]) if np.any(inds == i) else np.nan
        for i in range(1, len(edges))
    ])

    return centers, binned

# PLOT
fig, axes = plt.subplots(len(planets), 2, figsize=(12, 21))

for i, planet in enumerate(planets):

    c1, rbin = centroid_fold(
        time_clean, centr_row_norm,
        planet["period"], planet["t0"]
    )

    c2, cbin = centroid_fold(
        time_clean, centr_col_norm,
        planet["period"], planet["t0"]
    )

    ax1 = axes[i, 0]
    ax2 = axes[i, 1]

    ax1.plot(c1, rbin, lw=1.3)
    ax2.plot(c2, cbin, lw=1.3)

    ax1.axhline(0, ls="--", alpha=0.5)
    ax2.axhline(0, ls="--", alpha=0.5)

    ax1.axvline(0, ls="--", alpha=0.4)
    ax2.axvline(0, ls="--", alpha=0.4)

    ax1.set_title(f"{planet['label']} Row")
    ax2.set_title(f"{planet['label']} Col")

    ax1.set_xlabel("Phase")
    ax2.set_xlabel("Phase")

    ax1.set_ylabel("Centroid shift")
    ax2.set_ylabel("Centroid shift")

plt.tight_layout()
plt.show()

In [ ]:
# INDIVIDUAL TRANSIT CONSISTENCY TEST

rng = np.random.default_rng(42)

planets = [
    {"label": "planet_b", "period": 1.4721,  "t0": 2906.0012, "duration": 30.4/1440},
    {"label": "planet_c", "period": 2.4510,  "t0": 2906.5254, "duration": 36.0/1440},
    {"label": "planet_d", "period": 3.9238,  "t0": 2906.2632, "duration": 42.0/1440},
    {"label": "planet_e", "period": 6.3704,  "t0": 2905.5554, "duration": 48.0/1440},
    {"label": "planet_f", "period": 9.0776,  "t0": 2911.8898, "duration": 55.0/1440},
    {"label": "planet_g", "period": 12.8753, "t0": 2907.6576, "duration": 62.0/1440},
]

def stacked_fold(times, fluxes, transit_times, duration, period):

    phase_all = []
    flux_all = []

    for tc in transit_times:

        mask = np.abs(times - tc) <= (2 * duration)

        if np.any(mask):
            ph = (times[mask] - tc) / period
            phase_all.extend(ph)
            flux_all.extend(fluxes[mask])

    phase_all = np.array(phase_all)
    flux_all = np.array(flux_all)

    half_width = max(0.05, 2.5 * duration / period)

    edges = np.linspace(-half_width, half_width, 401)
    centers = 0.5 * (edges[:-1] + edges[1:])

    inds = np.digitize(phase_all, edges)

    binned = np.array([
        np.median(flux_all[inds == i]) if np.any(inds == i) else np.nan
        for i in range(1, len(edges))
    ])

    return centers, binned, half_width

fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

for k, p in enumerate(planets):

    ax = axes[k]

    period = p["period"]
    t0 = p["t0"]
    duration = p["duration"]

    n_start = int(np.floor((t_pld.min() - t0) / period))
    n_end   = int(np.ceil((t_pld.max() - t0) / period))

    n_vals = np.arange(n_start, n_end + 1)
    t_transits = t0 + n_vals * period

    valid = (t_transits >= t_pld.min()) & (t_transits <= t_pld.max())
    t_transits = t_transits[valid]

    if p["label"] in ["planet_b", "planet_c"] and len(t_transits) > 20:
        idx = rng.choice(len(t_transits), 20, replace=False)
        t_plot = np.sort(t_transits[idx])
    else:
        t_plot = t_transits

    # individual lines
    for tc in t_plot:

        mask = np.abs(t_pld - tc) <= (2 * duration)

        if np.any(mask):
            ph = (t_pld[mask] - tc) / period
            fw = f_pld[mask]

            ax.plot(ph, fw, color="gray", alpha=0.3, lw=0.8)

    # stacked fold
    c, bf, hw = stacked_fold(
        t_pld, f_pld, t_transits, duration, period
    )

    ax.plot(c, bf, lw=2.0, color="blue")

    ax.axhline(1.0, ls="--", alpha=0.5)
    ax.axvline(0.0, ls="--", alpha=0.4)

    ax.set_xlim(-hw, hw)
    ax.set_title(f"{p['label']}\nP={period:.4f} d")
    ax.set_xlabel("Phase")
    ax.set_ylabel("Flux")

for j in range(len(planets), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

For Diagnostic 6 check TLS

 Diagnostic-by-Diagnostic Conclusions
* Diagnostic 1 — Phase Fold:
All seven planets show clear transit dips centred at phase 0 with flat wings in the stacked phase fold. Planet h is marginal due to only ~4 transits. All other planets show unambiguous transit morphology.
* Diagnostic 2 — Odd/Even Transit Comparison:
Planets b and c pass cleanly under TLS periods. Planets d and e are marginal — depth differences present but consistent with noise given limited transit counts. Planet f is flagged — significant depth asymmetry between odd and even transits, though only 8 transits limits statistical confidence. Planets g and h have insufficient transits for meaningful assessment. CETRA periods produce unreliable odd/even results for all planets due to period offsets of 1.4–4.3% causing transit smearing.
* Diagnostic 3 — Secondary Eclipse Search:
All seven planets pass cleanly under both TLS and CETRA periods. No secondary eclipse detected at phase ±0.5 for any candidate. No evidence of background eclipsing binary contamination within the aperture. This is the strongest and most unambiguous diagnostic result across the entire Glance suite.
* Diagnostic 4 — Centroid Motion:
Inconclusive for all planets. Raw centroid columns (MOM_CENTR1/2, PSF_CENTR1/2) are absent from this TPF — consistent with the handover doc warning that PSF centroids are entirely unreliable for this faint red target. Flux-weighted centroids computed from the pixel cube showed noise levels of 0.1–0.3 pixels throughout the phase range — too large to identify transit-associated centroid shifts above the noise floor. This diagnostic cannot be performed reliably on this dataset and is noted as a known limitation.
* Diagnostic 5 — Individual Transit Inspection:
Planets b, c, d show consistent individual transit morphology under TLS periods — individual transits scatter symmetrically around the stacked median dip. Planet e is marginal — large scatter from few transits and period offset. Planet f is flagged — stacked median shows asymmetric shape with steep ingress and gradual egress, unusual for a planetary transit. Planet g and h are marginal due to insufficient transits. CETRA individual transits confirm that period precision is the dominant factor — CETRA's larger offsets cause complete transit smearing for planet b and near-complete smearing for c and d.

* Diagnostic 6 — Nearby Star Contamination:
Aperture of 11 pixels is well isolated around the TRAPPIST-1 PSF core. Total flux outside aperture is 7.17% — this is PSF wing loss, not contamination. Background artefact at column 0 rows 1-2 contributes flux equivalent to 16% of peak pixel but is fully excluded from the science aperture and does not enter the light curve. Its physical origin (background star vs detector artefact) should be verified against the Gillon et al. 2017 field description. No contaminating source identified within the science aperture.

# Phase 7 - BATMAN Transit Parameter Fitting

This is done only on TLS results as CETRA results were less accurate.  
From now on due to no detection Planet h will not be considered for any analysis.

In [ ]:
pip show batman-package

In [ ]:
# SEMI-MAJOR AXIS IN STELLAR RADII

# Constants
G = 6.674e-11
M_star = 0.0898   # solar masses
M_sun = 1.989e30
R_star = 0.1179   # solar radii
R_sun_m = 6.957e8

M_star_kg = M_star * M_sun
R_star_m = R_star * R_sun_m

# Planet periods (days)
planets = [
    ("b", 1.5109),
    ("c", 2.4219),
    ("d", 3.9279),
    ("e", 5.8910),
    ("f", 9.8162),
    ("g", 11.6624),
]

rows = []

for name, P_days in planets:

    P_seconds = P_days * 86400

    a_meters = (
        G * M_star_kg * P_seconds**2 / (4 * np.pi**2)
    ) ** (1/3)

    a_stellar = a_meters / R_star_m

    rows.append({
        "Planet": name,
        "Period_days": P_days,
        "a_meters": a_meters,
        "a_Rstar": a_stellar
    })

df = pd.DataFrame(rows)

pd.set_option("display.float_format", "{:.4f}".format)
print(df)

In [ ]:
# INITIAL PLANET RADIUS RATIOS (Rp/Rs)
# Using: Rp/Rs = sqrt(1 - depth)

# TLS depths (normalized in-transit flux level)
depths = {
    "b": 0.99007,
    "c": 0.99029,
    "d": 0.98926,
    "e": 0.98750,
    "f": 0.98251,
    "g": 0.98609,
}

rows = []

for planet, depth in depths.items():

    transit_depth = 1.0 - depth
    rp_rs = np.sqrt(transit_depth)

    rows.append({
        "Planet": planet,
        "TLS_depth": depth,
        "Transit_depth": transit_depth,
        "Rp_Rs": rp_rs
    })

df = pd.DataFrame(rows)

pd.set_option("display.float_format", "{:.6f}".format)
print(df.sort_values("Planet").reset_index(drop=True))

In [ ]:
# BATMAN + NELDER-MEAD FIT FOR ALL 6 PLANETS
import batman
from scipy.optimize import minimize

# PLANET TABLE
planets = [
    {"label":"b", "period":1.510881488162373, "t0":2906.5449,         "a_rs":21.0479, "rp_guess":0.09965, "duration":30.4/1440},
    {"label":"c", "period":2.421904078690552, "t0":2907.526796022453, "a_rs":28.8286, "rp_guess":0.09854, "duration":36.0/1440},
    {"label":"d", "period":3.927910492384035, "t0":2907.223378096564, "a_rs":39.7948, "rp_guess":0.10363, "duration":42.0/1440},
    {"label":"e", "period":5.8910,            "t0":2911.120669274915, "a_rs":52.1409, "rp_guess":0.11180, "duration":48.0/1440},
    {"label":"f", "period":9.8162,            "t0":2915.059895518406, "a_rs":73.2850, "rp_guess":0.13225, "duration":55.0/1440},
    {"label":"g", "period":11.6624,           "t0":2907.420936785209, "a_rs":82.2074, "rp_guess":0.11794, "duration":62.0/1440},
]

# BATMAN MODEL
def transit_model(times, period, t0, rp, a_rs):

    params = batman.TransitParams()
    params.t0 = t0
    params.per = period
    params.rp = rp
    params.a = a_rs
    params.inc = 89.5
    params.ecc = 0.0
    params.w = 90.0
    params.u = [0.4, 0.2]
    params.limb_dark = "quadratic"

    m = batman.TransitModel(
        params,
        times,
        supersample_factor=7,
        exp_time=2/1440   # 2-minute cadence
    )

    return m.light_curve(params)

# OBJECTIVE
def objective(x, times, flux, period, a_rs, t0_guess):

    rp, t0 = x

    if not (0.05 <= rp <= 0.20):
        return 1e9

    if not (t0_guess - 0.5 <= t0 <= t0_guess + 0.5):
        return 1e9

    model_flux = transit_model(times, period, t0, rp, a_rs)

    return np.sum((flux - model_flux)**2)

# STORAGE
fit_results = []

# LOOP
for p in planets:

    label    = p["label"]
    period   = p["period"]
    t0_guess = p["t0"]
    a_rs     = p["a_rs"]
    rp_guess = p["rp_guess"]
    dur      = p["duration"]

    print("="*70)
    print(f"Planet {label}")
    print("="*70)

    # USE 2-MIN BINNED DATA
    n_start = int(np.floor((t_tls_b.min() - t0_guess) / period))
    n_end   = int(np.ceil((t_tls_b.max() - t0_guess) / period))
    n_vals  = np.arange(n_start, n_end + 1)

    tc_all = t0_guess + n_vals * period
    tc_all = tc_all[(tc_all >= t_tls_b.min()) & (tc_all <= t_tls_b.max())]

    mask_total = np.zeros(len(t_tls_b), dtype=bool)

    for tc in tc_all:
        mask_total |= (np.abs(t_tls_b - tc) <= dur)

    t_fit = t_tls_b[mask_total]
    f_fit = f_tls_b[mask_total]

    print("Fit points:", len(t_fit))

    # OPTIMIZE
    x0 = [rp_guess, t0_guess]

    res = minimize(
        objective,
        x0=x0,
        args=(t_fit, f_fit, period, a_rs, t0_guess),
        method="Nelder-Mead",
        options={
            "maxiter": 3000,
            "xatol": 1e-6,
            "fatol": 1e-6
        }
    )

    rp_fit, t0_fit = res.x
    inc_fit = 89.5

    print("Success :", res.success)
    print("rp      :", rp_fit)
    print("t0      :", t0_fit)
    print("inc     :", inc_fit)
    print("chi2    :", res.fun)

    fit_results.append({
        "planet": label,
        "rp": float(rp_fit),
        "t0": float(t0_fit),
        "inc": float(inc_fit),
        "chi2": float(res.fun)
    })

    # PHASE PLOT (2-min binned data)
    phase = ((t_tls_b - t0_fit) / period) % 1
    phase[phase > 0.5] -= 1

    show = np.abs(phase) <= 0.03

    t_model = np.linspace(t0_fit - 0.05*period, t0_fit + 0.05*period, 1000)
    f_model = transit_model(t_model, period, t0_fit, rp_fit, a_rs)

    phase_model = ((t_model - t0_fit) / period) % 1
    phase_model[phase_model > 0.5] -= 1

    plt.figure(figsize=(8,4))
    plt.plot(phase[show], f_tls_b[show], ".", ms=3, alpha=0.45)
    plt.plot(phase_model, f_model, lw=2)

    plt.axvline(0, ls="--", alpha=0.5)
    plt.axhline(1.0, ls="--", alpha=0.4)

    plt.title(f"Planet {label} | rp={rp_fit:.4f}")
    plt.xlabel("Phase")
    plt.ylabel("Flux")
    plt.xlim(-0.03, 0.03)
    plt.tight_layout()
    plt.show()

print("\nFINAL FIT RESULTS")
for r in fit_results:
    print(r)

In [ ]:
# COMPUTE PLANET RADII FROM BATMAN FIT RESULTS

R_star_km = 81935.0
R_earth_km = 6371.0

# Published reference values (approx Earth radii)
published_radii = {
    "b": 1.12,
    "c": 1.10,
    "d": 0.79,
    "e": 0.92,
    "f": 1.05,
    "g": 1.13,
}

print(f"{'Planet':<8}{'rp/R*':>12}{'Radius (Re)':>16}{'Published (Re)':>18}{'Delta':>12}")

radius_results = []

for r in fit_results:

    label = r["planet"]
    rp = r["rp"]

    # Rp in km
    Rp_km = rp * R_star_km

    # Convert to Earth radii
    Rp_re = Rp_km / R_earth_km

    pub = published_radii.get(label, np.nan)
    delta = Rp_re - pub if np.isfinite(pub) else np.nan

    radius_results.append({
        "planet": label,
        "rp_rs": rp,
        "radius_re": Rp_re,
        "published_re": pub,
        "delta": delta
    })

    print(
        f"{label:<8}"
        f"{rp:>12.5f}"
        f"{Rp_re:>16.3f}"
        f"{pub:>18.3f}"
        f"{delta:>12.3f}"
    )

In [ ]:
# BATMAN MODEL DURATIONS vs TLS vs PUBLISHED

import pandas as pd
import batman

# PLANET TABLE
planets = [
    {"label":"b", "period":1.510881488162373, "t0":2906.5449,         "a_rs":21.0479, "duration":30.4},
    {"label":"c", "period":2.421904078690552, "t0":2907.526796022453, "a_rs":28.8286, "duration":36.0},
    {"label":"d", "period":3.927910492384035, "t0":2907.223378096564, "a_rs":39.7948, "duration":42.0},
    {"label":"e", "period":5.8910,            "t0":2911.120669274915, "a_rs":52.1409, "duration":48.0},
    {"label":"f", "period":9.8162,            "t0":2915.059895518406, "a_rs":73.2850, "duration":55.0},
    {"label":"g", "period":11.6624,           "t0":2907.420936785209, "a_rs":82.2074, "duration":62.0},
]

# PUBLISHED DURATIONS (minutes)
published = {
    "b": 36.0,
    "c": 42.0,
    "d": 49.0,
    "e": 57.0,
    "f": 63.0,
    "g": 68.0
}

# lookup fitted rp
rp_lookup = {r["planet"]: r["rp"] for r in fit_results}

rows = []

for p in planets:

    label  = p["label"]
    period = p["period"]
    t0     = p["t0"]
    a_rs   = p["a_rs"]
    rp     = rp_lookup[label]

    # batman params
    params = batman.TransitParams()
    params.t0 = t0
    params.per = period
    params.rp = rp
    params.a = a_rs
    params.inc = 89.5
    params.ecc = 0.0
    params.w = 90.0
    params.u = [0.4, 0.2]
    params.limb_dark = "quadratic"

    # dense one-period model
    t_model = np.linspace(t0 - period/2, t0 + period/2, 10000)

    m = batman.TransitModel(
        params,
        t_model,
        supersample_factor=7,
        exp_time=2/1440
    )

    flux_model = m.light_curve(params)

    # threshold crossing
    in_transit = flux_model < 0.9999

    if np.any(in_transit):
        t_ingress = t_model[np.where(in_transit)[0][0]]
        t_egress  = t_model[np.where(in_transit)[0][-1]]
        duration_min = (t_egress - t_ingress) * 1440.0
    else:
        duration_min = np.nan

    rows.append({
        "Planet": label,
        "Model_duration_min": duration_min,
        "TLS_duration_min": p["duration"],
        "Published_min": published[label],
        "Diff_vs_TLS_min": duration_min - p["duration"],
        "Diff_vs_Published_min": duration_min - published[label]
    })

df = pd.DataFrame(rows)

print(df.round(2).to_string(index=False))

In [ ]:
# BATMAN MODEL DURATIONS vs TLS vs PUBLISHED

import pandas as pd
import batman

# PLANET TABLE
planets = [
    {"label":"b", "period":1.510881488162373, "t0":2906.5449,         "a_rs":21.0479, "duration":30.4},
    {"label":"c", "period":2.421904078690552, "t0":2907.526796022453, "a_rs":28.8286, "duration":36.0},
    {"label":"d", "period":3.927910492384035, "t0":2907.223378096564, "a_rs":39.7948, "duration":42.0},
    {"label":"e", "period":5.8910,            "t0":2911.120669274915, "a_rs":52.1409, "duration":48.0},
    {"label":"f", "period":9.8162,            "t0":2915.059895518406, "a_rs":73.2850, "duration":55.0},
    {"label":"g", "period":11.6624,           "t0":2907.420936785209, "a_rs":82.2074, "duration":62.0},
]

# PUBLISHED DURATIONS (minutes)
published = {
    "b": 36.0,
    "c": 42.0,
    "d": 49.0,
    "e": 57.0,
    "f": 63.0,
    "g": 68.0
}

# lookup fitted rp
rp_lookup = {r["planet"]: r["rp"] for r in fit_results}

rows = []

for p in planets:

    label  = p["label"]
    period = p["period"]
    t0     = p["t0"]
    a_rs   = p["a_rs"]
    rp     = rp_lookup[label]

    # batman params
    params = batman.TransitParams()
    params.t0 = t0
    params.per = period
    params.rp = rp
    params.a = a_rs
    params.inc = 89.5
    params.ecc = 0.0
    params.w = 90.0
    params.u = [0.4, 0.2]
    params.limb_dark = "quadratic"

    # dense one-period model
    t_model = np.linspace(t0 - period/2, t0 + period/2, 10000)

    m = batman.TransitModel(
        params,
        t_model,
        supersample_factor=7,
        exp_time=2/1440
    )

    flux_model = m.light_curve(params)

    # threshold crossing
    in_transit = flux_model < 0.9999

    if np.any(in_transit):
        t_ingress = t_model[np.where(in_transit)[0][0]]
        t_egress  = t_model[np.where(in_transit)[0][-1]]
        duration_min = (t_egress - t_ingress) * 1440.0
    else:
        duration_min = np.nan

    rows.append({
        "Planet": label,
        "Model_duration_min": duration_min,
        "TLS_duration_min": p["duration"],
        "Published_min": published[label],
        "Diff_vs_TLS_min": duration_min - p["duration"],
        "Diff_vs_Published_min": duration_min - published[label]
    })

df = pd.DataFrame(rows)

print(df.round(2).to_string(index=False))

Phase 5 used the Mandel-Agol transit model with the batman (v2.5.3) package to fit transits for six TRAPPIST-1 planets (b–g). Planet h was excluded because its detected depth was inconsistent with known values, indicating a likely systematic false signal.

The fitting process used fixed orbital parameters (zero eccentricity, inclination 89.5°, quadratic limb darkening) while optimising only planet radius ratio (rp) and mid-transit time (t0) using SciPy minimisation on corrected light-curve data.

Main Results
* Planets b and c: Radius estimates closely matched published values, so these are considered reliable.
* Planets d–g: Radii were overestimated by 50–80%, mainly due to period errors causing transit misalignment during stacking. These values are treated as upper limits.
* Transit durations: Model-derived durations were more accurate than TLS estimates, especially for inner planets (b, c, d), matching literature within about 1 minute.  

Batman fitting successfully recovered physically sensible transit parameters for all six planets, with strong results for inner planets and useful upper-limit estimates for outer planets.

# Phase 8 - TTV Analysis

In [ ]:
# PREDICTED TRANSIT TIMES WITHIN DATA BASELINE

t0_fit_lookup = {r["planet"]: r["t0"] for r in fit_results}

print("EXPECTED TRANSITS WITHIN OBSERVED BASELINE")

baseline_start = t_pld.min()
baseline_end   = t_pld.max()

all_transits = {}

for p in planets:

    label  = p["label"]
    period = p["period"]
    t0_fit = t0_fit_lookup[label]

    # integer transit indices spanning baseline
    n_start = int(np.floor((baseline_start - t0_fit) / period))
    n_end   = int(np.ceil((baseline_end   - t0_fit) / period))

    n_vals = np.arange(n_start, n_end + 1)

    # predicted linear ephemeris
    t_pred = t0_fit + n_vals * period

    # keep only those inside observed baseline
    keep = (t_pred >= baseline_start) & (t_pred <= baseline_end)

    n_vals = n_vals[keep]
    t_pred = t_pred[keep]

    all_transits[label] = {
        "n": n_vals,
        "times": t_pred
    }

    print(
        f"Planet {label:>2s} | "
        f"P={period:8.5f} d | "
        f"t0_fit={t0_fit:11.6f} | "
        f"Expected transits = {len(t_pred):2d}"
    )

print(f"Baseline start: {baseline_start:.6f}")
print(f"Baseline end  : {baseline_end:.6f}")

In [ ]:
# INDIVIDUAL TRANSIT TIMING FITS
from scipy.optimize import minimize_scalar

# LOOKUP TABLES
fit_lookup = {r["planet"]: r for r in fit_results}

# BATMAN MODEL WITH ONLY t0 FREE
def model_t0_only(times, t0_trial, period, rp, a_rs, inc):

    params = batman.TransitParams()
    params.t0 = t0_trial
    params.per = period
    params.rp = rp
    params.a = a_rs
    params.inc = inc
    params.ecc = 0.0
    params.w = 90.0
    params.u = [0.4, 0.2]
    params.limb_dark = "quadratic"

    m = batman.TransitModel(
        params,
        times,
        supersample_factor=7,
        exp_time=1/1440
    )

    return m.light_curve(params)

# STORAGE
timing_results = []

print("INDIVIDUAL TRANSIT TIMING FITS")

baseline_start = t_pld.min()
baseline_end   = t_pld.max()

for p in planets:

    label    = p["label"]
    period   = p["period"]
    a_rs     = p["a_rs"]
    dur      = p["duration"]          # days
    fitrow   = fit_lookup[label]

    rp       = fitrow["rp"]
    inc      = fitrow["inc"]
    t0_ref   = fitrow["t0"]

    print(f"\nPlanet {label}")

    # predicted transit sequence
    n_start = int(np.floor((baseline_start - t0_ref) / period))
    n_end   = int(np.ceil((baseline_end   - t0_ref) / period))

    n_vals = np.arange(n_start, n_end + 1)
    t_pred = t0_ref + n_vals * period

    keep = (t_pred >= baseline_start) & (t_pred <= baseline_end)

    n_vals = n_vals[keep]
    t_pred = t_pred[keep]

    kept_count = 0

    for n, tc in zip(n_vals, t_pred):

        # narrow fitting window
        half_window = 1.5 * (dur / 1440)
        mask = np.abs(t_pld - tc) <= half_window

        if np.sum(mask) < 5:
            continue

        t_win = t_pld[mask]
        f_win = f_pld[mask]

        # objective: fit only t0
        def objective(t0_trial):
            model_flux = model_t0_only(
                t_win, t0_trial, period, rp, a_rs, inc
            )
            return np.sum((f_win - model_flux)**2)

        # bounds ±0.5 duration
        res = minimize_scalar(
            objective,
            bounds=(tc - 0.5*(dur/1440), tc + 0.5*(dur/1440)),
            method="bounded",
            options={"xatol":1e-6}
        )

        t0_fit = res.x
        oc = t0_fit - tc

        timing_results.append({
            "planet": label,
            "n": int(n),
            "predicted_t0": float(tc),
            "fitted_t0": float(t0_fit),
            "O_minus_C_days": float(oc),
            "O_minus_C_min": float(oc * 1440.0),
            "points": int(len(t_win)),
            "success": bool(res.success)
        })

        kept_count += 1

    print(f"Predicted transits: {len(t_pred)} | Fit windows kept: {kept_count}")

print(f"Total fitted transits stored: {len(timing_results)}")

timing_df = pd.DataFrame(timing_results)
display(timing_df.head(20))

In [ ]:
# REFINE PERIODS FROM TRANSIT TIMINGS

rows = []
print("PERIOD REFINEMENT FROM TRANSIT TIMINGS")

# Loop over planets
for p in planets:

    label = p["label"]
    tls_period = p["period"]

    # isolate one planet
    sub = timing_df[timing_df["planet"] == label].copy()

    if len(sub) < 2:
        print(f"{label}: insufficient transits")
        continue

    # sort by transit number
    sub = sub.sort_values("n")

    # arrays
    n_arr = sub["n"].values.astype(float)
    fitted_t0_arr = sub["fitted_t0"].values.astype(float)


    # Linear ephemeris fit:
    # fitted_t0 = P*n + T0

    P_refined, T0_refined = np.polyfit(
        n_arr,
        fitted_t0_arr,
        1
    )

    # compare with TLS
    diff_days = P_refined - tls_period
    diff_min = diff_days * 1440.0

    # store back into planets list
    p["period_tls"] = tls_period
    p["period"] = P_refined
    p["T0_refined"] = T0_refined

    rows.append({
        "Planet": label,
        "TLS_period_d": tls_period,
        "Refined_period_d": P_refined,
        "Difference_min": diff_min
    })

    print(
        f"{label:2s} | "
        f"TLS={tls_period:.10f} d | "
        f"Refined={P_refined:.10f} d | "
        f"Δ={diff_min:.4f} min"
    )

# Summary table
period_refined_df = pd.DataFrame(rows)

print("\nSUMMARY TABLE")
print(period_refined_df.round(8).to_string(index=False))

In [ ]:
# STEP 3 — O-C (TTV) PLOTS FOR ALL 6 PLANETS

timing_df = pd.DataFrame(timing_results)

planet_order = ["b", "c", "d", "e", "f", "g"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharey=False)
axes = axes.flatten()

for i, planet in enumerate(planet_order):

    ax = axes[i]

    sub = timing_df[timing_df["planet"] == planet].copy()
    sub = sub.sort_values("n")

    if len(sub) == 0:
        ax.text(0.5, 0.5, "No timings",
                ha="center", va="center",
                transform=ax.transAxes)
        ax.set_title(f"Planet {planet}")
        continue

    ax.plot(
        sub["n"],
        sub["O_minus_C_min"],
        "o-",
        ms=4,
        lw=1.2,
        alpha=0.85
    )

    # zero line = linear ephemeris
    ax.axhline(0, ls="--", lw=1, alpha=0.6)

    rms = np.std(sub["O_minus_C_min"])

    ax.set_title(
        f"Planet {planet}\n"
        f"N={len(sub)} | RMS={rms:.2f} min"
    )

    ax.set_xlabel("Transit number n")
    ax.set_ylabel("O-C (minutes)")

for j in range(len(planet_order), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

These TTV signals are physically consistent with known TRAPPIST-1 TTV behaviour from published literature. The amplitudes and patterns match expectations from the resonance chain. This is a strong validation that your PLD-corrected light curve and individual transit fitting are working correctly.

In [ ]:
# LOMB-SCARGLE PERIOD SEARCH ON O-C DATA
from astropy.timeseries import LombScargle

# DATAFRAME
timing_df = pd.DataFrame(timing_results)

planet_order = ["b", "c", "d", "e", "f", "g"]

results_ls = []

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, planet in enumerate(planet_order):

    ax = axes[i]

    sub = timing_df[timing_df["planet"] == planet].copy()
    sub = sub.sort_values("predicted_t0")

    t = sub["predicted_t0"].values              # days
    y = sub["O_minus_C_min"].values            # minutes

    npts = len(sub)

    # Too few transits: report amplitude only
    if planet in ["f", "g"] or npts < 8:

        amp = np.nanmax(y) - np.nanmin(y) if npts > 0 else np.nan

        results_ls.append({
            "planet": planet,
            "n_transits": npts,
            "peak_period_days": np.nan,
            "peak_power": np.nan,
            "peak_to_peak_min": amp,
            "note": "Too few transits; period unconstrained"
        })

        ax.plot(t, y, "o-", ms=4)
        ax.axhline(0, ls="--", alpha=0.6)
        ax.set_title(
            f"Planet {planet}\n"
            f"N={npts} | p2p={amp:.2f} min"
        )
        ax.set_xlabel("Predicted t0 (days)")
        ax.set_ylabel("O-C (min)")
        continue

    # Lomb-Scargle
    baseline = t.max() - t.min()

    min_freq = 1.0 / baseline
    max_freq = 0.5              # ~2 day shortest modulation

    freq, power = LombScargle(t, y).autopower(
        minimum_frequency=min_freq,
        maximum_frequency=max_freq,
        samples_per_peak=10
    )

    best_idx = np.argmax(power)
    best_freq = freq[best_idx]
    best_period = 1.0 / best_freq

    orbital_periods = [1.511, 2.422, 3.928, 5.891, 9.816, 11.662]
    for op in orbital_periods:
        if abs(best_period - op) / op < 0.20:
            print(f"  WARNING: period {best_period:.2f}d close to orbital period {op:.3f}d")

    amp = np.nanmax(y) - np.nanmin(y)

    results_ls.append({
        "planet": planet,
        "n_transits": npts,
        "peak_period_days": best_period,
        "peak_power": power[best_idx],
        "peak_to_peak_min": amp,
        "note": "LS peak"
    })

    # plot periodogram
    ax.plot(1.0 / freq, power, lw=1.2)
    ax.axvline(best_period, ls="--", alpha=0.7)

    ax.set_title(
        f"Planet {planet}\n"
        f"Pttv={best_period:.1f} d"
    )
    ax.set_xlabel("TTV Period (days)")
    ax.set_ylabel("Power")

# clean extras
for j in range(len(planet_order), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

ls_df = pd.DataFrame(results_ls)

print("\nLomb-Scargle TTV Summary")
print(ls_df.round(3).to_string(index=False))

In [ ]:
# DETREND O-C BY REMOVING LINEAR PERIOD DRIFT

planets_order = ["b", "c", "d", "e", "f", "g"]


all_rows = []

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

for i, pl in enumerate(planets_order):

    ax = axes[i]

    sub = timing_df[timing_df["planet"] == pl].copy()

    # require enough points for fit
    if len(sub) < 2:
        ax.text(0.5, 0.5, "Not enough transits",
                ha="center", va="center")
        ax.set_title(f"Planet {pl}")
        continue

    # sort by transit number
    sub = sub.sort_values("n").reset_index(drop=True)

    # arrays
    n_arr = sub["n"].values.astype(float)
    oc_arr = sub["O_minus_C_min"].values.astype(float)

    # Step 2: linear fit   O-C = slope*n + intercept
    slope, intercept = np.polyfit(n_arr, oc_arr, 1)

    # Step 3: reconstruct trend
    trend = slope * n_arr + intercept

    # Step 4: detrend
    oc_det = oc_arr - trend
    sub["O_minus_C_detrended"] = oc_det

    # Step 5: peak-to-peak amplitude
    ptp_raw = np.max(oc_arr) - np.min(oc_arr)
    ptp_det = np.max(oc_det) - np.min(oc_det)

    print("=" * 70)
    print(f"Planet {pl}")
    print(f"N transits      : {len(sub)}")
    print(f"Slope (min/tr)  : {slope:.4f}")
    print(f"Intercept (min) : {intercept:.4f}")
    print(f"Raw P2P (min)   : {ptp_raw:.3f}")
    print(f"Detr P2P (min)  : {ptp_det:.3f}")

    # Step 6: plot raw + detrended
    ax.plot(n_arr, oc_arr, "o-", lw=1.5, ms=5, label="Raw O-C")
    ax.plot(n_arr, oc_det, "s--", lw=1.5, ms=4, label="Detrended")

    ax.axhline(0, ls="--", alpha=0.5)

    ax.set_title(
        f"Planet {pl}\n"
        f"Slope={slope:.3f} min/transit"
    )
    ax.set_xlabel("Transit number n")
    ax.set_ylabel("O-C (min)")
    ax.legend(fontsize=8)

    all_rows.append(sub)

# hide unused panels
for j in range(len(planets_order), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

# combine into updated dataframe
timing_df_detrended = pd.concat(all_rows, ignore_index=True)

print("\nCreated dataframe: timing_df_detrended")
print("New column added: O_minus_C_detrended")

In [ ]:
# FIRST-ORDER TTV MASS ESTIMATES FROM RESONANCE SCALING

# CONSTANTS
M_star_solar = 0.0898
M_earth_in_solar = 3.003e-6

# stellar mass in Earth masses
M_star_earth = M_star_solar / M_earth_in_solar

# PERIOD LOOKUP
P = {p["label"]: p["period"] for p in planets}

# DETRENDED PEAK-TO-PEAK VALUES (minutes)
p2p_det_min = {
    "b": 32.69,
    "c": 10.59,
    "d": 43.08,
    "e": 45.22,
    "f": 21.59,
    "g": 51.73,
}

amp_lookup = {
    k: 0.5 * v / 1440.0
    for k, v in p2p_det_min.items()
}

# RESONANCE PAIRS
pairs = [
    {"planet":"b", "perturber":"c", "inner":"b", "outer":"c", "j":5},
    {"planet":"c", "perturber":"b", "inner":"b", "outer":"c", "j":5},

    {"planet":"d", "perturber":"e", "inner":"d", "outer":"e", "j":2},
    {"planet":"e", "perturber":"d", "inner":"d", "outer":"e", "j":2},

    {"planet":"f", "perturber":"g", "inner":"f", "outer":"g", "j":3},
    {"planet":"g", "perturber":"f", "inner":"f", "outer":"g", "j":3},
]

# COMPUTE MASSES
rows = []

for pair in pairs:

    pl   = pair["planet"]
    pert = pair["perturber"]
    inn  = pair["inner"]
    out  = pair["outer"]
    j    = pair["j"]

    P_inner  = P[inn]
    P_outer  = P[out]
    P_planet = P[pl]

    # resonance offset
    Delta = abs(P_outer / P_inner - j / (j - 1)) * (j - 1)

    # semi-amplitude in days
    dt_days = amp_lookup[pl]

    # first-order scaling mass estimate
    M_pert_earth = (dt_days / P_planet) * (Delta / j) * M_star_earth

    rows.append({
        "Planet": pl,
        "Primary_perturber": pert,
        "j": j,
        "P_planet_d": P_planet,
        "Delta": Delta,
        "Detrended_P2P_min": p2p_det_min[pl],
        "TTV_semiamp_min": dt_days * 1440.0,
        "Estimated_perturber_Mearth": M_pert_earth
    })

mass_df = pd.DataFrame(rows)

print("RESONANCE TTV MASS ESTIMATES (DETRENDED)")
print(mass_df.round(4).to_string(index=False))

# Phase 9 - Orbital and Physical Properties

In [ ]:
# KEPLERIAN SEMI-MAJOR AXES FROM TLS PERIODS

# Constants
M_star = 0.0898   # solar masses

# Compute a using Kepler's Third Law
# For units:
# P in years
# a in AU
# M in solar masses

# a^3 = M_star * P^2
# a = (M_star * P^2)^(1/3)

rows = []

for p in planets:

    label = p["label"]
    P_days = p["period"]

    P_years = P_days / 365.25

    # semi-major axis in AU
    a_au = (M_star * P_years**2)**(1/3)

    p["a_au"] = a_au

    rows.append({
        "Planet": label,
        "Period_days": P_days,
        "Period_years": P_years,
        "a_AU": a_au
    })

# results table
a_df = pd.DataFrame(rows)

print("SEMI-MAJOR AXES USING TLS PERIODS")
print(a_df.round(6).to_string(index=False))

In [ ]:
# COMPUTE STELLAR MASS FROM a AND P USING SI UNITS

# Constants
G = 6.674e-11          # m^3 kg^-1 s^-2
AU_M = 1.496e11        # metres
M_SUN = 1.989e30       # kg

rows = []

for p in planets:

    label = p["label"]
    P_days = p["period"]
    a_au = p["a_au"]

    # Convert units
    a_m = a_au * AU_M
    P_s = P_days * 86400.0

    # Kepler rearranged:
    # M = 4*pi^2*a^3 / (G*P^2)
    M_kg = (4 * np.pi**2 * a_m**3) / (G * P_s**2)
    M_solar = M_kg / M_SUN

    # store
    p["M_star_fit_solar"] = M_solar

    rows.append({
        "Planet": label,
        "Period_days": P_days,
        "a_AU": a_au,
        "a_m": a_m,
        "Period_s": P_s,
        "M_star_solar": M_solar
    })

mass_check_df = pd.DataFrame(rows)

print("STELLAR MASS RECOVERED FROM TLS a AND P")
print(mass_check_df.round(6).to_string(index=False))

In [ ]:
# ORBITAL VELOCITY FROM a AND PERIOD

# Constants
AU_M = 1.496e11   # metres per AU

rows = []

for p in planets:

    label = p["label"]
    a_au = p["a_au"]
    P_days = p["period"]

    # Unit conversions
    a_m = a_au * AU_M
    P_s = P_days * 86400.0

    # Circular orbital speed:
    # v = 2*pi*a / P
    v_ms = 2 * np.pi * a_m / P_s
    v_kms = v_ms / 1000.0

    # store for later use
    p["v_kms"] = v_kms

    rows.append({
        "Planet": label,
        "a_AU": a_au,
        "v_kms": v_kms
    })

vel_df = pd.DataFrame(rows)

print("ORBITAL VELOCITIES FROM TLS PERIODS")
print(vel_df.round(6).to_string(index=False))

In [ ]:
# INCIDENT STELLAR FLUX IN EARTH UNITS

# Constants
L_star_solar = 0.000553
L_sun = 3.828e26          # W
AU_M = 1.496e11           # m
S_earth = 1361.0          # W m^-2

# stellar luminosity in watts
L_star_W = L_star_solar * L_sun

rows = []

for p in planets:

    label = p["label"]
    a_au = p["a_au"]

    # Convert orbital distance to metres

    a_m = a_au * AU_M

    # Flux at orbit:
    # F = L / (4*pi*a^2)
    # Convert to Earth units by dividing by solar constant

    flux_wm2 = L_star_W / (4 * np.pi * a_m**2)
    flux_earth = flux_wm2 / S_earth

    # store
    p["flux_earth"] = flux_earth

    rows.append({
        "Planet": label,
        "a_AU": a_au,
        "Flux_EarthUnits": flux_earth
    })

flux_df = pd.DataFrame(rows)

print("INCIDENT STELLAR FLUX (EARTH = 1)")
print(flux_df.round(6).to_string(index=False))

In [ ]:
# EQUILIBRIUM TEMPERATURES FOR PLANETS

# Constants
T_star = 2566.0                 # K
R_star_solar = 0.1179          # solar radii
SOLAR_RADIUS_TO_AU = 0.00465   # AU per solar radius
albedo = 0.30

# stellar radius in AU
R_star_AU = R_star_solar * SOLAR_RADIUS_TO_AU

rows = []

for p in planets:

    label = p["label"]
    a_au = p["a_au"]
    flux_earth = p["flux_earth"]


    # Equilibrium temperature:
    # T_eq = T_star * sqrt(R_star / (2a)) * (1-A)^(1/4)

    T_eq = (
        T_star
        * np.sqrt(R_star_AU / (2.0 * a_au))
        * (1.0 - albedo)**0.25
    )

    # store
    p["T_eq"] = T_eq

    rows.append({
        "Planet": label,
        "a_AU": a_au,
        "Flux_EarthUnits": flux_earth,
        "T_eq_K": T_eq
    })

teq_df = pd.DataFrame(rows)

print("PLANET EQUILIBRIUM TEMPERATURES")
print(teq_df.round(6).to_string(index=False))

In [ ]:
# BULK DENSITIES USING PIPELINE TTV MASSES + BATMAN RADII

# Build adopted mass lookup from mass_df
adopted_masses = {
    row["Primary_perturber"]: row["Estimated_perturber_Mearth"]
    for _, row in mass_df.iterrows()
}

EARTH_DENSITY_GCC = 5.51

# Update planets with Rp_re from radius_results
for p in planets:
    for r_res in radius_results:
        if p["label"] == r_res["planet"]:
            p["Rp_re"] = r_res["radius_re"]
            break

rows = []

for p in planets:

    label = p["label"]

    if label not in adopted_masses:
        continue

    M_earth = adopted_masses[label]
    Rp_re = p["Rp_re"]

    # Volume in Earth-radius units
    V = (4.0 / 3.0) * np.pi * Rp_re**3

    # density relative to Earth
    rho_earth = M_earth / V

    # convert to g/cm^3
    rho_gcc = rho_earth * EARTH_DENSITY_GCC

    # store
    p["adopted_mass_me"] = M_earth
    p["rho_earth"] = rho_earth
    p["rho_gcc"] = rho_gcc

    rows.append({
        "Planet": label,
        "M_earth": M_earth,
        "R_earth": Rp_re,
        "rho_earth": rho_earth,
        "rho_gcc": rho_gcc
    })

density_df = pd.DataFrame(rows)

print("PLANET BULK DENSITIES (Pipeline TTV masses + Batman radii)")
print(density_df.round(4).to_string(index=False))

The scatter across six orders of magnitude tell us that both error sources, overestimated TTV masses and overestimated batman radii, are compounding unpredictably. For some planets they partially cancel, for others they amplify.  The 79-day K2 baseline is insufficient to recover reliable masses from first-order resonance scaling, and period offsets of 3-7% for outer planets propagate into unreliable radii.

In [ ]:
# ESCAPE VELOCITY IN EARTH UNITS

EARTH_ESCAPE_KMS = 11.186

rows = []

for p in planets:

    label = p["label"]

    if ("adopted_mass_me" not in p) or ("Rp_re" not in p):
        continue

    M_earth = p["adopted_mass_me"]
    R_earth = p["Rp_re"]

    # Escape velocity (km/s)
    v_esc = EARTH_ESCAPE_KMS * np.sqrt(M_earth / R_earth)

    # store back into planet dict
    p["v_esc_kms"] = v_esc

    rows.append({
        "Planet": label,
        "M_earth": M_earth,
        "R_earth": R_earth,
        "v_esc_kms": v_esc
    })

escape_df = pd.DataFrame(rows)

print("PLANET ESCAPE VELOCITIES")
print(escape_df.round(4).to_string(index=False))

In [ ]:
# EARTH SIMILARITY INDEX (ESI)

# Earth reference values and weights
earth_ref = {
    "radius":   {"value": 1.0,    "weight": 0.57},
    "density":  {"value": 5.51,   "weight": 1.07},
    "velocity": {"value": 11.186, "weight": 0.70},
    "temp":     {"value": 288.0,  "weight": 5.58},
}

rows = []

for p in planets:

    label = p["label"]

    required = ["Rp_re", "rho_gcc", "adopted_mass_me", "T_eq"]

    if not all(k in p for k in required):
        continue

    # retrieve values
    R = p["Rp_re"]
    rho = p["rho_gcc"]
    M = p["adopted_mass_me"]
    T = p["T_eq"]

    # compute escape velocity (km/s)
    v_esc = 11.186 * np.sqrt(M / R)

    # store
    p["v_esc_kms"] = v_esc

    # ESI formula:
    # ESI_i = (1 - |(x-x0)/(x+x0)|)^w

    ESI_radius = (
        1 - abs((R - earth_ref["radius"]["value"]) /
                (R + earth_ref["radius"]["value"]))
    ) ** earth_ref["radius"]["weight"]

    ESI_density = (
        1 - abs((rho - earth_ref["density"]["value"]) /
                (rho + earth_ref["density"]["value"]))
    ) ** earth_ref["density"]["weight"]

    ESI_velocity = (
        1 - abs((v_esc - earth_ref["velocity"]["value"]) /
                (v_esc + earth_ref["velocity"]["value"]))
    ) ** earth_ref["velocity"]["weight"]

    ESI_temp = (
        1 - abs((T - earth_ref["temp"]["value"]) /
                (T + earth_ref["temp"]["value"]))
    ) ** earth_ref["temp"]["weight"]

    # total ESI
    ESI_total = (
        ESI_radius *
        ESI_density *
        ESI_velocity *
        ESI_temp
    )

    rows.append({
        "Planet": label,
        "ESI_radius": ESI_radius,
        "ESI_density": ESI_density,
        "ESI_velocity": ESI_velocity,
        "ESI_temp": ESI_temp,
        "ESI_total": ESI_total
    })

esi_df = pd.DataFrame(rows)

print("EARTH SIMILARITY INDEX (ESI)")
print(esi_df.round(4).to_string(index=False))

In [ ]:
# HABITABLE ZONE CLASSIFICATION (Kopparapu+2013)

def classify_hz(flux):

    if flux > 1.78:
        return "Too hot"

    elif 1.08 < flux <= 1.78:
        return "Optimistic HZ inner"

    elif 0.36 <= flux <= 1.08:
        return "Conservative HZ"

    elif 0.20 <= flux < 0.36:
        return "Optimistic HZ outer"

    else:
        return "Too cold"

# Apply classification to planets
for p in planets:

    if "flux_earth" not in p:
        continue

    p["HZ_status"] = classify_hz(p["flux_earth"])

# Quick check
print("HABITABLE ZONE CLASSIFICATION")
for p in planets:

    if "HZ_status" in p:
        print(
            f"{p['label']:>2s} | "
            f"Flux = {p['flux_earth']:.3f} S⊕ | "
            f"{p['HZ_status']}"
        )

# Phase 10 - Final Summary

In [ ]:
# FINAL SUMMARY TABLE

# keep only needed ESI column
esi_lookup = esi_df[["Planet", "ESI_total"]]

rows = []

for p in planets:

    rows.append({

        "Planet": p["label"],

        "Period_days":
            p.get("period", np.nan),

        "Rp_re":
            p.get("Rp_re", np.nan),

        "adopted_mass_me":
            p.get("adopted_mass_me", np.nan),

        "a_AU":
            p.get("a_au", np.nan),

        "v_kms":
            p.get("v_kms", np.nan),

        "flux_earth":
            p.get("flux_earth", np.nan),

        "T_eq":
            p.get("T_eq", np.nan),

        "rho_gcc":
            p.get("rho_gcc", np.nan),

        "HZ_status":
            p.get("HZ_status", "Unknown")
    })

summary_df = pd.DataFrame(rows)

# merge ESI_total
summary_df = summary_df.merge(
    esi_lookup,
    on="Planet",
    how="left"
)

# reorder columns
summary_df = summary_df[[
    "Planet",
    "Period_days",
    "Rp_re",
    "adopted_mass_me",
    "a_AU",
    "v_kms",
    "flux_earth",
    "T_eq",
    "rho_gcc",
    "ESI_total",
    "HZ_status"
]]

print("FINAL PLANET SUMMARY TABLE")
print(summary_df.round(4).to_string(index=False))